In [2]:


#add these files before running the ntoebook.
'''
 llps_plus.xlxs
 llps_minus.xlsx
 protein_plus.xlsx
 protein_minus.xlsx
 Data.zip
 iupred3.py
 iupred3_lib.py
 '''


#


'\n llps_plus.xlxs\n llps_minus.xlsx\n protein_plus.xlsx\n protein_minus.xlsx\n Data.zip\n iupred3.py\n iupred3_lib.py\n '

In [3]:
import pandas as pd
import numpy as np

#Load Datasets'

llps_pos = pd.read_excel("llps_plus.xlsx")
llps_neg = pd.read_excel("llps_minus.xlsx")

protein_pos = pd.read_excel("protein_plus.xlsx")
protein_neg = pd.read_excel("protein_minus.xlsx")

print(f"LLPS positive:  {llps_pos.shape}")
print(f"LLPS negative:  {llps_neg.shape}")
print(f"Protein positive: {protein_pos.shape}")
print(f"Protein negative: {protein_neg.shape}")




llps = pd.concat(
    [llps_pos, llps_neg],
    ignore_index=True
)

protein = pd.concat(
    [protein_pos, protein_neg],
    ignore_index=True
)

llps["PSID"] = llps["PSID"].astype(str).str.strip()

print(f"\nCombined LLPS dataset: {llps.shape}")
print(f"Combined protein dataset: {protein.shape}")



before = len(llps)

llps = llps[
    llps["Components type"]
    .astype(str)
    .str.strip()
    .str.lower()
    == "protein(1)"
].copy()

print(
    f"\nComponents type == protein(1): "
    f"{len(llps)} / {before} rows kept"
)



before = len(llps)

llps = llps[
    llps["Protein type (N/D)"]
    .astype(str)
    .str.strip()
    .str.upper()
    == "N"
].copy()

print(
    f"Protein type (N/D) == N: "
    f"{len(llps)} / {before} rows kept"
)


#clean protein ID

llps["Protein ID"] = (
    llps["Protein ID"]
    .astype(str)
    .str.strip()
)

protein["PID"] = (
    protein["PID"]
    .astype(str)
    .str.strip()
)


# basic summary

print("\nFinal filtered LLPS dataset:")
print(llps.shape)

print("\nUnique Protein IDs:")
print(llps["Protein ID"].nunique())

print("\nComponents type:")
print(llps["Components type"].value_counts())

print("\nProtein type:")
print(llps["Protein type (N/D)"].value_counts())

LLPS positive:  (4111, 31)
LLPS negative:  (1894, 31)
Protein positive: (504, 20)
Protein negative: (351, 20)

Combined LLPS dataset: (6005, 31)
Combined protein dataset: (855, 20)

Components type == protein(1): 3266 / 6005 rows kept
Protein type (N/D) == N: 2869 / 3266 rows kept

Final filtered LLPS dataset:
(2869, 31)

Unique Protein IDs:
282

Components type:
Components type
protein(1)     2736
Protein(1)      108
protein(1)       25
Name: count, dtype: int64

Protein type:
Protein type (N/D)
N    2869
Name: count, dtype: int64


In [4]:
# create LLPS Label

print("Unique values in 'Phase separation':")
print(llps["Phase separation"].value_counts(dropna=False))


llps["Phase separation"] = (
    llps["Phase separation"]
    .astype(str)
    .str.strip()
    .str.lower()
)


label_map = {
    "yes": 1,
    "no": 0
}

llps["llps_label"] = llps["Phase separation"].map(label_map)


unmapped = llps[llps["llps_label"].isna()]

if len(unmapped) > 0:
    print("\nWARNING: Unrecognized label values found:")
    print(unmapped["Phase separation"].value_counts(dropna=False))
else:
    print("\nAll labels successfully mapped.")


# Remove rows without a valid label
before = len(llps)

llps = llps.dropna(subset=["llps_label"]).copy()

llps["llps_label"] = llps["llps_label"].astype(int)

print(f"\nRemoved {before - len(llps)} rows with invalid/missing labels.")


# Final label summary
print("\nFinal label distribution:")
print(llps["llps_label"].value_counts())

print("\nLabel proportions:")
print(llps["llps_label"].value_counts(normalize=True))

Unique values in 'Phase separation':
Phase separation
Yes    1798
No     1071
Name: count, dtype: int64

All labels successfully mapped.

Removed 0 rows with invalid/missing labels.

Final label distribution:
llps_label
1    1798
0    1071
Name: count, dtype: int64

Label proportions:
llps_label
1    0.626699
0    0.373301
Name: proportion, dtype: float64


In [5]:
import re
import pandas as pd


def clean_sequence(sequence):
    """Extract and clean a protein sequence from a FASTA entry."""
    if pd.isna(sequence):
        return None

    lines = str(sequence).strip().splitlines()

    if lines and lines[0].startswith(">"):
        lines = lines[1:]

    sequence = "".join(lines)
    sequence = re.sub(r"[^A-Za-z]", "", sequence).upper()

    return sequence or None


llps["Sequence_clean"] = llps["Sequence"].apply(clean_sequence)


# Check for non-standard amino acids
standard_amino_acids = set("ACDEFGHIKLMNPQRSTVWY")

llps["has_nonstandard_aa"] = llps["Sequence_clean"].apply(
    lambda seq: seq is not None and not set(seq).issubset(standard_amino_acids)
)

print("\nSequence QC:")
print("Missing sequences:", llps["Sequence_clean"].isna().sum())
print(
    "Sequences with non-standard amino acids:",
    llps["has_nonstandard_aa"].sum()
)


# Identify non-standard residues
nonstandard_counts = {}

for seq in llps.loc[llps["Sequence_clean"].notna(), "Sequence_clean"]:
    for aa in set(seq) - standard_amino_acids:
        nonstandard_counts[aa] = nonstandard_counts.get(aa, 0) + 1

print("\nNon-standard amino acids found:")
print(nonstandard_counts)


# Remove invalid sequences
before = len(llps)

llps = llps[
    llps["Sequence_clean"].notna()
    & ~llps["has_nonstandard_aa"]
].copy()

llps.reset_index(drop=True, inplace=True)

print(f"\nRemoved {before - len(llps)} rows with invalid sequences.")
print(f"Remaining rows: {len(llps)}")


# Sequence length statistics
llps["sequence_length"] = llps["Sequence_clean"].str.len()

print("\nSequence length statistics:")
print(llps["sequence_length"].describe())

print("\nExample sequences:")
print(
    llps[
        ["Protein ID", "Protein name", "Sequence_clean", "sequence_length"]
    ].head()
)


Sequence QC:
Missing sequences: 6
Sequences with non-standard amino acids: 22

Non-standard amino acids found:
{'X': 13, 'U': 17, 'O': 17, 'B': 6}

Removed 28 rows with invalid sequences.
Remaining rows: 2841

Sequence length statistics:
count    2841.000000
mean      343.808518
std       231.859801
min        16.000000
25%       168.000000
50%       317.000000
75%       441.000000
max      1748.000000
Name: sequence_length, dtype: float64

Example sequences:
  Protein ID Protein name                                     Sequence_clean  \
0      p0001        LAF-1  MESNQSNNGGSGNAALNRGGRYVPPHLRGGDGGAAAAASAGGDDRR...   
1      p0001        LAF-1  MESNQSNNGGSGNAALNRGGRYVPPHLRGGDGGAAAAASAGGDDRR...   
2      p0001      LAF-1_m  ESNQSNNGGSGNAALNRGGRYVPPHLRGGDGGAAAAASAGGDDRRG...   
3      p0001      LAF-1_m  ESNQSNNGGSGNAALNRGGRYVPPHLRGGDGGAAAAASAGGDDRRG...   
4      p0001      LAF-1_m  ESNQSNNGGSGNAALNRGGRYVPPHLRGGDGGAAAAASAGGDDRRG...   

   sequence_length  
0              708  
1           

In [6]:
TEMP_MIN_C = 0.0
TEMP_MAX_C = 60.0

NON_QUANTITATIVE = {
    "freeze/thaw",
    "freezethaw",
    "freeze-thaw",
    "warmbyhand",
    "warmingbyhand",
    "frozen",
}


def normalize_temperature_text(text):
    if pd.isna(text):
        return None

    text = str(text).strip()

    if not text or text.upper() in {"-", "NA", "N/A", "NAN"}:
        return None

    if text.upper() == "RT":
        return "25C"

    text = (
        text.replace("℃", "C")
        .replace("°C", "C")
        .replace("˚C", "C")
        .replace("ºC", "C")
        .replace("–", "-")
        .replace("−", "-")
        .replace("－", "-")
        .replace("＜", "<")
        .replace("＞", ">")
        .replace("≤", "<=")
        .replace("≥", ">=")
    )

    return re.sub(r"\s+", "", text)


def temperature_to_celsius(value, unit):
    value = float(value)

    if unit.upper() == "K":
        return value - 273.15

    return value


def extract_signed_numbers(text):
    numbers = []
    i = 0

    while i < len(text):
        if text[i].isdigit():
            j = i
            while j < len(text) and (text[j].isdigit() or text[j] == "."):
                j += 1

            numbers.append(float(text[i:j]))
            i = j

        elif text[i] == "-":
            is_negative = i == 0 or text[i - 1] == "-"

            if is_negative and i + 1 < len(text) and text[i + 1].isdigit():
                j = i + 1
                while j < len(text) and (text[j].isdigit() or text[j] == "."):
                    j += 1

                numbers.append(-float(text[i + 1:j]))
                i = j
            else:
                i += 1

        else:
            i += 1

    return numbers


def extract_process_note(text):
    match = re.search(r"\(([^)]*)\)", text)

    if not match:
        return text, None, np.nan

    note = match.group(1)
    remaining = text[:match.start()] + text[match.end():]
    note_lower = note.lower()

    if "cooling" in note_lower:
        direction = "cooling"
    elif "heating" in note_lower:
        direction = "heating"
    else:
        direction = note

    rate_match = re.search(
        r"(\d+(?:\.\d+)?)\s*[A-Za-z]*\s*/\s*min",
        note
    )
    rate = float(rate_match.group(1)) if rate_match else np.nan

    return remaining, direction, rate


def parse_temperature_range(text):
    text = normalize_temperature_text(text)

    if text is None:
        return np.nan, np.nan, "missing", None, np.nan

    if text.lower() in NON_QUANTITATIVE:
        return np.nan, np.nan, "non_quantitative", None, np.nan

    text, process, ramp_rate = extract_process_note(text)

    unit_matches = re.findall(r"[CK]", text.upper())
    unit = unit_matches[-1] if unit_matches else "C"

    numeric_text = re.sub(r"[CK]", "", text, flags=re.I)

    uncertainty_match = re.search(
        r"(-?\d+(?:\.\d+)?)\s*±\s*(\d+(?:\.\d+)?)",
        numeric_text
    )

    if uncertainty_match:
        central = temperature_to_celsius(
            uncertainty_match.group(1), unit
        )
        uncertainty = float(uncertainty_match.group(2))

        if numeric_text.startswith((">", ">=")):
            return (
                central,
                max(TEMP_MAX_C, central),
                "open_high_with_uncertainty",
                process,
                ramp_rate
            )

        if numeric_text.startswith(("<", "<=")):
            return (
                min(TEMP_MIN_C, central),
                central,
                "open_low_with_uncertainty",
                process,
                ramp_rate
            )

        return (
            central - uncertainty,
            central + uncertainty,
            "range_with_uncertainty",
            process,
            ramp_rate
        )

    inequality_matches = re.findall(
        r"(<=|>=|<|>)\s*(-?\d+(?:\.\d+)?)",
        numeric_text
    )

    if inequality_matches:
        low = TEMP_MIN_C
        high = TEMP_MAX_C

        for comparator, value in inequality_matches:
            value_c = temperature_to_celsius(value, unit)

            if comparator in (">", ">="):
                low = value_c
            else:
                high = value_c

        if len(inequality_matches) == 1:
            comparator, value = inequality_matches[0]
            value_c = temperature_to_celsius(value, unit)

            if comparator in (">", ">="):
                return (
                    value_c,
                    max(TEMP_MAX_C, value_c),
                    "open_high",
                    process,
                    ramp_rate
                )

            return (
                min(TEMP_MIN_C, value_c),
                value_c,
                "open_low",
                process,
                ramp_rate
            )

        if low > high:
            return (
                np.nan,
                np.nan,
                "outside_sampling_limit",
                process,
                ramp_rate
            )

        return low, high, "inequality_range", process, ramp_rate

    values = extract_signed_numbers(numeric_text)

    if not values:
        return np.nan, np.nan, "unparsed", process, ramp_rate

    values = [temperature_to_celsius(value, unit) for value in values]

    if len(values) == 1:
        value = values[0]

        if TEMP_MIN_C <= value <= TEMP_MAX_C:
            status = "single"
        else:
            status = "single_outside_limit"

        return value, value, status, process, ramp_rate

    low = min(values)
    high = max(values)
    status = "range" if len(values) == 2 else "multi_value_range"

    return low, high, status, process, ramp_rate


temperature_parsed = (
    llps["Temperature"]
    .apply(parse_temperature_range)
    .apply(pd.Series)
)

temperature_parsed.columns = [
    "temperature_min_C",
    "temperature_max_C",
    "temperature_parse_status",
    "temperature_process",
    "temperature_ramp_rate_C_per_min"
]

llps = pd.concat([llps, temperature_parsed], axis=1)


print("\nTemperature parsing status:")
print(llps["temperature_parse_status"].value_counts(dropna=False))

print("\nTemperature examples:")
print(
    llps[
        [
            "Temperature",
            "temperature_min_C",
            "temperature_max_C",
            "temperature_parse_status"
        ]
    ]
    .drop_duplicates()
    .head(50)
)

problem_status = {
    "missing",
    "non_quantitative",
    "unparsed",
    "ambiguous_range",
    "outside_sampling_limit"
}

print("\nMissing / non-quantitative / problematic temperatures:")
print(
    llps.loc[
        llps["temperature_parse_status"].isin(problem_status),
        [
            "Temperature",
            "temperature_min_C",
            "temperature_max_C",
            "temperature_parse_status"
        ]
    ]
    .drop_duplicates()
    .head(100)
)

outside_limit = llps[
    llps["temperature_min_C"].notna()
    & (
        (llps["temperature_min_C"] < TEMP_MIN_C)
        | (llps["temperature_max_C"] > TEMP_MAX_C)
    )
]

print(
    f"\nTemperatures extending outside "
    f"{TEMP_MIN_C}–{TEMP_MAX_C} °C: {len(outside_limit)} rows"
)

print(
    outside_limit[
        [
            "Temperature",
            "temperature_min_C",
            "temperature_max_C",
            "temperature_parse_status"
        ]
    ]
    .drop_duplicates()
    .head(50)
)


Temperature parsing status:
temperature_parse_status
single                       2632
range                         116
open_high                      34
open_low                       25
open_low_with_uncertainty      15
non_quantitative                7
single_outside_limit            5
missing                         4
inequality_range                2
multi_value_range               1
Name: count, dtype: int64

Temperature examples:
    Temperature  temperature_min_C  temperature_max_C temperature_parse_status
0            RT               25.0              25.00                   single
2          ＜15℃                0.0              15.00                 open_low
3          ＜37℃                0.0              37.00                 open_low
4          ＜33℃                0.0              33.00                 open_low
5          ＜40℃                0.0              40.00                 open_low
6           25℃               25.0              25.00                   single
10  

In [7]:
PH_MIN = 0.0
PH_MAX = 14.0

BUFFER_PH_LOOKUP = [
    (re.compile(r"\bcsf-?xb\b", re.I), 7.7, "literature_buffer_default"),
    (re.compile(r"\bbrb-?80\b", re.I), 6.8, "literature_buffer_default"),
    (
        re.compile(
            r"\bdpbs\b|dulbecco.?s?\s*phosphate\s*buffered\s*saline", re.I
        ),
        7.4,
        "literature_buffer_default",
    ),
    (
        re.compile(r"\bpbs\b|phosphate\s*buffered\s*saline", re.I),
        7.4,
        "literature_buffer_default",
    ),
    (re.compile(r"\bbis-?tris\b", re.I), 6.5, "typical_buffer_pH_estimate"),
    (
        re.compile(r"k2hpo4\s*/\s*kh2po4|potassium\s*phosphate", re.I),
        7.0,
        "typical_buffer_pH_estimate",
    ),
    (re.compile(r"\bmops\b", re.I), 7.2, "typical_buffer_pH_estimate"),
    (re.compile(r"\bmes\b", re.I), 6.0, "typical_buffer_pH_estimate"),
    (re.compile(r"\bpipes\b", re.I), 6.8, "typical_buffer_pH_estimate"),
    (re.compile(r"\bhepes\b", re.I), 7.4, "typical_buffer_pH_estimate"),
    (re.compile(r"\btris(?:[-/]hcl)?\b", re.I), 7.5, "typical_buffer_pH_estimate"),
    (re.compile(r"\bacetate\b", re.I), 4.8, "typical_buffer_pH_estimate"),
]

NO_PH_PATTERNS = re.compile(
    r"\bwater\b|\bdmso\b|\bcell\s*extract\b|\boxygen\s*scavenger",
    re.I
)


def normalize_pH_text(text):
    if pd.isna(text):
        return None

    text = str(text).strip()

    if not text or text.upper() in {"-", "NA", "N/A", "NAN"}:
        return None

    text = (
        text.replace("＜", "<")
        .replace("＞", ">")
        .replace("≤", "<=")
        .replace("≥", ">=")
        .replace("–", "-")
        .replace("−", "-")
        .replace("~", "-")
        .replace("：", ":")
        .replace("＝", "=")
    )

    return re.sub(r"\s+", " ", text).strip()


def parse_pH_range(text):
    text = normalize_pH_text(text)

    if text is None:
        return np.nan, np.nan, "missing"

    if re.search(
        r"\bpH\s*=?\s*\(?\s*(NA|N/?A|unknown)\s*\)?",
        text,
        flags=re.I
    ) or text.lower() == "unknown":
        return np.nan, np.nan, "explicit_missing"

    # These experiments involve a change in pH rather than one fixed value.
    if re.search(
        r"\bpH\b.*\b(?:jump|shift|changed|change)\b",
        text,
        flags=re.I
    ):
        return np.nan, np.nan, "dynamic_pH"

    # Multiple discrete pH conditions
    match = re.search(
        r"\bpH\s*[\(\[]\s*"
        r"(\d+(?:\.\d+)?(?:\s*,\s*\d+(?:\.\d+)?)+)"
        r"\s*[\)\]]",
        text,
        flags=re.I
    )

    if match:
        values = [float(x.strip()) for x in match.group(1).split(",")]

        if all(PH_MIN <= value <= PH_MAX for value in values):
            return min(values), max(values), "multiple_discrete"

        return np.nan, np.nan, "outside_limit"

    # Separate alternative ranges such as "3-4 or 10-11"
    if re.search(r"\bpH\b.*\bor\b", text, flags=re.I):
        ranges = re.findall(
            r"(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)",
            text
        )

        if len(ranges) >= 2:
            return np.nan, np.nan, "multiple_ranges"

    # pH from X to Y
    match = re.search(
        r"\bpH\s*(?:=|:)?\s*from\s*"
        r"(\d+(?:\.\d+)?)\s*(?:to|-)\s*"
        r"(\d+(?:\.\d+)?)",
        text,
        flags=re.I
    )

    if match:
        values = [float(match.group(1)), float(match.group(2))]
        low, high = min(values), max(values)

        if PH_MIN <= low <= PH_MAX and PH_MIN <= high <= PH_MAX:
            return low, high, "range"

        return np.nan, np.nan, "outside_limit"

    # Regular pH ranges
    match = re.search(
        r"\bpH\s*(?:=|:)?\s*[\(\[]?\s*"
        r"(\d+(?:\.\d+)?)\s*-\s*"
        r"(\d+(?:\.\d+)?)\s*[\)\]]?",
        text,
        flags=re.I
    )

    if match:
        values = [float(match.group(1)), float(match.group(2))]
        low, high = min(values), max(values)

        if PH_MIN <= low <= PH_MAX and PH_MIN <= high <= PH_MAX:
            return low, high, "range"

        return np.nan, np.nan, "outside_limit"

    # pH with uncertainty
    match = re.search(
        r"\bpH\s*(?:=|:)?\s*[\(\[]?\s*"
        r"(\d+(?:\.\d+)?)\s*(?:±|\+/-)\s*"
        r"(\d+(?:\.\d+)?)\s*[\)\]]?",
        text,
        flags=re.I
    )

    if match:
        center = float(match.group(1))
        uncertainty = abs(float(match.group(2)))

        low = center - uncertainty
        high = center + uncertainty

        if PH_MIN <= low <= PH_MAX and PH_MIN <= high <= PH_MAX:
            return low, high, "range_with_uncertainty"

        return np.nan, np.nan, "outside_limit"

    # pH inequalities
    match = re.search(
        r"\bpH\s*(?:=|:)?\s*[\(\[]?\s*"
        r"(<=|>=|<|>|≤|≥)\s*"
        r"(\d+(?:\.\d+)?)",
        text,
        flags=re.I
    )

    if match:
        comparator = (
            match.group(1)
            .replace("≤", "<=")
            .replace("≥", ">=")
        )
        value = float(match.group(2))

        if comparator in (">", ">="):
            return value, PH_MAX, "open_high"

        return PH_MIN, value, "open_low"

    # Exact pH
    match = re.search(
        r"\bpH\s*(?:=|:|at)?\s*[\(\[]?\s*"
        r"(\d+(?:\.\d+)?)\s*[\)\]]?",
        text,
        flags=re.I
    )

    if match:
        value = float(match.group(1))

        if PH_MIN <= value <= PH_MAX:
            return value, value, "single"

        return value, value, "single_outside_limit"

    # Buffer name followed directly by a pH value
    match = re.search(
        r"(?:tris(?:-hcl|/hcl)?|hepes|pipes|mes|mops|"
        r"phosphate(?:\s+buffer)?|acetate(?:\s+buffer)?)"
        r"\s*\(?\s*(\d+(?:\.\d+)?)\s*\)?",
        text,
        flags=re.I
    )

    if match:
        value = float(match.group(1))

        if PH_MIN <= value <= PH_MAX:
            return value, value, "single"

    if NO_PH_PATTERNS.search(text):
        return np.nan, np.nan, "no_defined_pH"

    for pattern, value, status in BUFFER_PH_LOOKUP:
        if pattern.search(text):
            return value, value, status

    return np.nan, np.nan, "unparsed"


pH_parsed = (
    llps["Buffer"]
    .apply(parse_pH_range)
    .apply(pd.Series)
)

pH_parsed.columns = [
    "pH_min",
    "pH_max",
    "pH_parse_status"
]

llps = pd.concat([llps, pH_parsed], axis=1)


print("\n" + "=" * 70)
print("pH PARSING STATUS")
print("=" * 70)
print(llps["pH_parse_status"].value_counts(dropna=False))


print("\nRepresentative pH examples:")
print(
    llps[
        ["Buffer", "pH_min", "pH_max", "pH_parse_status"]
    ]
    .drop_duplicates()
    .head(100)
)


problem_status = {
    "missing",
    "explicit_missing",
    "unparsed",
    "dynamic_pH",
    "multiple_discrete",
    "multiple_ranges",
    "no_defined_pH"
}

print("\nProblematic / special pH cases:")
print(
    llps.loc[
        llps["pH_parse_status"].isin(problem_status),
        ["Buffer", "pH_min", "pH_max", "pH_parse_status"]
    ]
    .drop_duplicates()
    .head(150)
)


print("\nInferred (not explicitly stated) pH values:")
print(
    llps.loc[
        llps["pH_parse_status"].isin({
            "literature_buffer_default",
            "typical_buffer_pH_estimate"
        }),
        ["Buffer", "pH_min", "pH_max", "pH_parse_status"]
    ]
    .drop_duplicates()
)


pH_outside = llps[
    llps["pH_min"].notna()
    & (
        (llps["pH_min"] < PH_MIN)
        | (llps["pH_max"] > PH_MAX)
    )
]

print(
    f"\nRows with pH extending outside "
    f"{PH_MIN}-{PH_MAX}: {len(pH_outside)}"
)

print(
    pH_outside[
        ["Buffer", "pH_min", "pH_max", "pH_parse_status"]
    ]
    .drop_duplicates()
    .head(50)
)


pH PARSING STATUS
pH_parse_status
single                        2505
literature_buffer_default       88
missing                         70
typical_buffer_pH_estimate      69
range                           49
explicit_missing                34
no_defined_pH                    9
dynamic_pH                       9
open_low                         4
multiple_discrete                2
multiple_ranges                  2
Name: count, dtype: int64

Representative pH examples:
                                                Buffer  pH_min  pH_max  \
0                         20 mM Tris, pH 7.4, 1 mM DTT     7.4     7.4   
2                                   20 mM Tris, pH 7.5     7.5     7.5   
16     50 mM Tris/HCl, 2.5% Glycerol, 1 mM DTT, pH 7.4     7.4     7.4   
24                         20 mM Tris pH 7.4, 1 mM DTT     7.4     7.4   
30                  100mM K2HPO4/KH2PO4 buffer at pH 7     7.0     7.0   
..                                                 ...     ...     ...   
414    

In [8]:
MAX_PROTEIN_CONC_uM = 1000.0
CONC_SAMPLES = 10
CONC_OPEN_FACTOR = 10.0
AVG_RESIDUE_MASS_DA = 110.0

MOLAR_UNIT_RE = r"(mM|uM|µM|μM|nM|M)"
MASS_UNIT_RE = (
    r"(mg\s*/\s*m[lL]|mg\s*m[lL]\s*[-−]\s*1|"
    r"g\s*/\s*L|g\s*L\s*[-−]\s*1|"
    r"[µu]g\s*/\s*[µu][lL]|%)"
)

MASS_TO_MGML = {
    "mgml": 1.0,
    "gl": 1.0,
    "ugul": 1.0,
    "%": 10.0,
}


def normalize_mass_unit(unit):
    unit = unit.lower().replace(" ", "").replace("µ", "u").replace("−", "-")

    if unit.startswith("mg"):
        return "mgml"
    if unit.startswith("ug"):
        return "ugul"
    if unit.startswith("g"):
        return "gl"
    if unit == "%":
        return "%"

    return None


def convert_concentration_to_uM(value, unit):
    unit = unit.lower().replace("μ", "u").replace("µ", "u").strip()

    conversion = {
        "m": 1e6,
        "mm": 1e3,
        "um": 1.0,
        "u": 1.0,
        "nm": 1e-3,
    }

    if unit not in conversion:
        return np.nan

    return float(value) * conversion[unit]


def mass_concentration_to_uM(value_mgml, seq_length):
    if seq_length is None or pd.isna(seq_length) or seq_length <= 0:
        return np.nan

    molecular_weight = float(seq_length) * AVG_RESIDUE_MASS_DA
    return value_mgml / molecular_weight * 1e6


def parse_protein_concentration(text, seq_length=None):
    if pd.isna(text):
        return np.nan, np.nan, "missing"

    text = str(text).strip()

    if not text or text.upper() in {"-", "NA", "N/A", "NAN"}:
        return np.nan, np.nan, "missing"

    first = text.split(";")[0].strip()

    if re.search(r"unknow", first, flags=re.I):
        return np.nan, np.nan, "missing"

    if not re.search(r"\d", first):
        return np.nan, np.nan, "missing"

    # Molar concentration
    match = re.search(
        r"([<>≤≥]?\s*\d+(?:\.\d+)?)\s*(mM|uM|µM|μM|nM|M)?"
        r"\s*(?:-|–|−|~|to)\s*"
        r"(\d+(?:\.\d+)?)\s*(mM|uM|µM|μM|nM|M)\b",
        first,
        flags=re.I,
    )

    if match:
        value_a = float(re.sub(r"[<>≤≥\s]", "", match.group(1)))
        unit_a = match.group(2) or match.group(4)
        value_b = float(match.group(3))
        unit_b = match.group(4)

        value_a_uM = convert_concentration_to_uM(value_a, unit_a)
        value_b_uM = convert_concentration_to_uM(value_b, unit_b)

        if np.isnan(value_a_uM) or np.isnan(value_b_uM):
            return np.nan, np.nan, "unparsed"

        return min(value_a_uM, value_b_uM), max(value_a_uM, value_b_uM), "range"

    # Open-ended lower bound
    match = re.search(
        r"(>=|>|≥)\s*(\d+(?:\.\d+)?)"
        r"\s*(mM|uM|µM|μM|nM|M)\b",
        first,
        flags=re.I,
    )

    if match:
        value = float(match.group(2))
        unit = match.group(3)
        value_uM = convert_concentration_to_uM(value, unit)

        if np.isnan(value_uM):
            return np.nan, np.nan, "unparsed"

        max_value = min(
            MAX_PROTEIN_CONC_uM,
            value_uM * CONC_OPEN_FACTOR,
        )
        max_value = max(max_value, value_uM)

        return value_uM, max_value, "open_high"

    # Open-ended upper bound
    match = re.search(
        r"(<=|<|≤|＜)\s*(\d+(?:\.\d+)?)"
        r"\s*(mM|uM|µM|μM|nM|M)\b",
        first,
        flags=re.I,
    )

    if match:
        value = float(match.group(2))
        unit = match.group(3)
        value_uM = convert_concentration_to_uM(value, unit)

        if np.isnan(value_uM):
            return np.nan, np.nan, "unparsed"

        min_value = max(0.0, value_uM / CONC_OPEN_FACTOR)

        return min_value, value_uM, "open_low"

    # Single molar value
    match = re.search(
        r"(\d+(?:\.\d+)?)"
        r"\s*(mM|uM|µM|μM|nM|M)\b",
        first,
        flags=re.I,
    )

    if match:
        value = float(match.group(1))
        unit = match.group(2)
        value_uM = convert_concentration_to_uM(value, unit)

        if np.isnan(value_uM):
            return np.nan, np.nan, "unparsed"

        return value_uM, value_uM, "single"

    # Mass concentration
    match = re.search(
        r"([<>≤≥]?\s*\d+(?:\.\d+)?)\s*(?:-|–|−|~|to)\s*"
        r"(\d+(?:\.\d+)?)\s*" + MASS_UNIT_RE,
        first,
        flags=re.I,
    )

    if match:
        value_a = float(re.sub(r"[<>≤≥\s]", "", match.group(1)))
        value_b = float(match.group(2))
        unit = normalize_mass_unit(match.group(3))

        if unit is None:
            return np.nan, np.nan, "unparsed"

        value_a_uM = mass_concentration_to_uM(
            value_a * MASS_TO_MGML[unit],
            seq_length,
        )
        value_b_uM = mass_concentration_to_uM(
            value_b * MASS_TO_MGML[unit],
            seq_length,
        )

        if np.isnan(value_a_uM) or np.isnan(value_b_uM):
            return np.nan, np.nan, "unparsed"

        return min(value_a_uM, value_b_uM), max(value_a_uM, value_b_uM), "range_mw_est"

    # Open-ended lower bound
    match = re.search(
        r"(>=|>|≥)\s*(\d+(?:\.\d+)?)\s*" + MASS_UNIT_RE,
        first,
        flags=re.I,
    )

    if match:
        unit = normalize_mass_unit(match.group(3))

        if unit is None:
            return np.nan, np.nan, "unparsed"

        value_uM = mass_concentration_to_uM(
            float(match.group(2)) * MASS_TO_MGML[unit],
            seq_length,
        )

        if np.isnan(value_uM):
            return np.nan, np.nan, "unparsed"

        max_value = max(
            min(MAX_PROTEIN_CONC_uM, value_uM * CONC_OPEN_FACTOR),
            value_uM,
        )

        return value_uM, max_value, "open_high_mw_est"

    # Open-ended upper bound
    match = re.search(
        r"(<=|<|≤)\s*(\d+(?:\.\d+)?)\s*" + MASS_UNIT_RE,
        first,
        flags=re.I,
    )

    if match:
        unit = normalize_mass_unit(match.group(3))

        if unit is None:
            return np.nan, np.nan, "unparsed"

        value_uM = mass_concentration_to_uM(
            float(match.group(2)) * MASS_TO_MGML[unit],
            seq_length,
        )

        if np.isnan(value_uM):
            return np.nan, np.nan, "unparsed"

        min_value = max(0.0, value_uM / CONC_OPEN_FACTOR)

        return min_value, value_uM, "open_low_mw_est"

    # Single mass concentration
    match = re.search(
        r"(\d+(?:\.\d+)?)\s*" + MASS_UNIT_RE,
        first,
        flags=re.I,
    )

    if match:
        unit = normalize_mass_unit(match.group(2))

        if unit is None:
            return np.nan, np.nan, "unparsed"

        value_uM = mass_concentration_to_uM(
            float(match.group(1)) * MASS_TO_MGML[unit],
            seq_length,
        )

        if np.isnan(value_uM):
            return np.nan, np.nan, "unparsed"

        return value_uM, value_uM, "single_mw_est"

    return np.nan, np.nan, "unparsed"


protein_conc_parsed = (
    llps.apply(
        lambda row: parse_protein_concentration(
            row["Solute concentration"],
            row.get("sequence_length"),
        ),
        axis=1,
    )
    .apply(pd.Series)
)

protein_conc_parsed.columns = [
    "protein_conc_min_uM",
    "protein_conc_max_uM",
    "protein_conc_parse_status",
]

existing_columns = [
    col for col in protein_conc_parsed.columns
    if col in llps.columns
]

llps = llps.drop(columns=existing_columns)
llps = pd.concat([llps, protein_conc_parsed], axis=1)


print("\n" + "=" * 70)
print("PROTEIN CONCENTRATION PARSING")
print("=" * 70)

print(
    llps["protein_conc_parse_status"]
    .value_counts(dropna=False)
)

print("\nProtein concentration examples:")
print(
    llps[
        [
            "Solute concentration",
            "protein_conc_min_uM",
            "protein_conc_max_uM",
            "protein_conc_parse_status",
        ]
    ]
    .drop_duplicates()
    .head(80)
)

print("\nUnparsed protein concentrations:")
print(
    llps.loc[
        llps["protein_conc_parse_status"].isin({"missing", "unparsed"}),
        [
            "Solute concentration",
            "protein_conc_parse_status",
        ],
    ]
    .drop_duplicates()
    .head(100)
)


PROTEIN CONCENTRATION PARSING
protein_conc_parse_status
single              2174
range                278
single_mw_est        219
missing               80
open_high             47
unparsed              14
open_low              13
range_mw_est          11
open_high_mw_est       3
open_low_mw_est        2
Name: count, dtype: int64

Protein concentration examples:
    Solute concentration  protein_conc_min_uM  protein_conc_max_uM  \
0         3-5 µM [LAF-1]                  3.0                  5.0   
1         3.5 µM [LAF-1]                  3.5                  3.5   
2      12 µM [LAF-1 RGG]                 12.0                 12.0   
3      6 µM [LAF-1 RGG]2                  6.0                  6.0   
4      3 µM [LAF-1 RGG]2                  3.0                  3.0   
..                   ...                  ...                  ...   
168          20 µM [Tau]                 20.0                 20.0   
169          5 µM [Tau]                   5.0                  5.0   
172 

In [9]:
SALT_LIMITS = {
    "NaCl": 2000.0,
    "KCl": 1000.0,
    "MgCl2": 50.0,
}

SALT_MOLAR_MASS = {
    "NaCl": 58.44,
    "KCl": 74.55,
    "MgCl2": 95.21,
}

MOLAR_UNIT_RE = r"(mM|M|uM|nM)"
MASS_UNIT_RE = r"(g\s*/\s*L|mg\s*/\s*mL)"


def normalize_salt_text(text):
    if pd.isna(text):
        return None

    text = str(text).strip()

    if not text or text.upper() in {"-", "NA", "N/A", "NAN"}:
        return None

    text = (
        text.replace("μ", "u")
        .replace("µ", "u")
        .replace("–", "-")
        .replace("−", "-")
        .replace("＜", "<")
        .replace("＞", ">")
        .replace("≤", "<=")
        .replace("≥", ">=")
    )

    # Treat mol/kg as approximately mol/L for dilute solutions.
    text = re.sub(r"mol\s*/\s*kg", "M", text, flags=re.I)
    return re.sub(r"\s+", " ", text).strip()


def convert_salt_to_mM(value, unit):
    unit = unit.lower().replace("μ", "u").replace("µ", "u").replace(" ", "")

    factors = {
        "m": 1000.0,
        "mm": 1.0,
        "um": 0.001,
        "nm": 0.000001,
    }

    if unit not in factors:
        return np.nan

    return float(value) * factors[unit]


def convert_mass_to_mM(value, unit, salt_name):
    unit = unit.lower().replace(" ", "")
    molar_mass = SALT_MOLAR_MASS[salt_name]

    if unit in {"g/l", "gl", "mg/ml", "mgml"}:
        grams_per_litre = value
    else:
        return np.nan

    return grams_per_litre / molar_mass * 1000.0


def parse_one_salt(text, salt_name):
    text = normalize_salt_text(text)

    if text is None:
        return 0.0, 0.0, "absent"

    salt_pattern = re.escape(salt_name)

    # Range
    match = re.search(
        rf"(\d+(?:\.\d+)?)\s*{MOLAR_UNIT_RE}?\s*-\s*"
        rf"(\d+(?:\.\d+)?)\s*{MOLAR_UNIT_RE}\s*"
        rf"{salt_pattern}\b",
        text,
        flags=re.I,
    )

    if match:
        unit_a = match.group(2) or match.group(4)
        value_a = convert_salt_to_mM(float(match.group(1)), unit_a)
        value_b = convert_salt_to_mM(float(match.group(3)), match.group(4))

        if np.isnan(value_a) or np.isnan(value_b):
            return np.nan, np.nan, "unparsed"

        return min(value_a, value_b), max(value_a, value_b), "range"

    # Open-ended lower bound
    match = re.search(
        rf"(>=|>|≥)\s*(\d+(?:\.\d+)?)\s*"
        rf"{MOLAR_UNIT_RE}\s*{salt_pattern}\b",
        text,
        flags=re.I,
    )

    if match:
        value = convert_salt_to_mM(float(match.group(2)), match.group(3))

        if np.isnan(value):
            return np.nan, np.nan, "unparsed"

        max_value = max(
            value,
            min(SALT_LIMITS[salt_name], value * 10.0),
        )

        return value, max_value, "open_high"

    # Open-ended upper bound
    match = re.search(
        rf"(<=|<|≤)\s*(\d+(?:\.\d+)?)\s*"
        rf"{MOLAR_UNIT_RE}\s*{salt_pattern}\b",
        text,
        flags=re.I,
    )

    if match:
        value = convert_salt_to_mM(float(match.group(2)), match.group(3))

        if np.isnan(value):
            return np.nan, np.nan, "unparsed"

        return max(0.0, value / 10.0), value, "open_low"

    # Single value: "140 mM NaCl"
    match = re.search(
        rf"(\d+(?:\.\d+)?)\s*"
        rf"{MOLAR_UNIT_RE}\s*{salt_pattern}\b",
        text,
        flags=re.I,
    )

    if match:
        value = convert_salt_to_mM(float(match.group(1)), match.group(2))

        if np.isnan(value):
            return np.nan, np.nan, "unparsed"

        return value, value, "single"

    # Reversed order: "NaCl 140 mM"
    match = re.search(
        rf"{salt_pattern}\s+"
        rf"(\d+(?:\.\d+)?)\s*{MOLAR_UNIT_RE}\b",
        text,
        flags=re.I,
    )

    if match:
        value = convert_salt_to_mM(float(match.group(1)), match.group(2))

        if np.isnan(value):
            return np.nan, np.nan, "unparsed"

        return value, value, "single"

    # Mass concentration: "8 g/L NaCl"
    match = re.search(
        rf"(\d+(?:\.\d+)?)\s*"
        rf"{MASS_UNIT_RE}\s*{salt_pattern}\b",
        text,
        flags=re.I,
    )

    if match:
        value = convert_mass_to_mM(
            float(match.group(1)),
            match.group(2),
            salt_name,
        )

        if np.isnan(value):
            return np.nan, np.nan, "unparsed"

        return value, value, "single"

    if re.search(rf"\b{salt_pattern}\b", text, flags=re.I):
        return np.nan, np.nan, "unparsed"

    return 0.0, 0.0, "absent"


def parse_salts(text):
    results = {}

    for salt in ["NaCl", "KCl", "MgCl2"]:
        min_value, max_value, status = parse_one_salt(text, salt)

        results[f"{salt}_min_mM"] = min_value
        results[f"{salt}_max_mM"] = max_value
        results[f"{salt}_parse_status"] = status

    return results


salt_parsed = (
    llps["Salt concentration"]
    .apply(parse_salts)
    .apply(pd.Series)
)

existing_columns = [
    col for col in salt_parsed.columns
    if col in llps.columns
]

llps = llps.drop(columns=existing_columns)
llps = pd.concat([llps, salt_parsed], axis=1)


print("\n" + "=" * 70)
print("SALT PARSING STATUS")
print("=" * 70)

for salt in ["NaCl", "KCl", "MgCl2"]:
    print(f"\n{salt}:")
    print(
        llps[f"{salt}_parse_status"]
        .value_counts(dropna=False)
    )


print("\nRepresentative salt examples:")
print(
    llps[
        [
            "Salt concentration",
            "NaCl_min_mM",
            "NaCl_max_mM",
            "KCl_min_mM",
            "KCl_max_mM",
            "MgCl2_min_mM",
            "MgCl2_max_mM",
        ]
    ]
    .drop_duplicates()
    .head(100)
)


for salt in ["NaCl", "KCl", "MgCl2"]:
    problematic = llps[
        llps[f"{salt}_parse_status"] == "unparsed"
    ]

    print(f"\nUnparsed {salt} entries:")
    print(
        problematic[
            [
                "Salt concentration",
                f"{salt}_min_mM",
                f"{salt}_max_mM",
                f"{salt}_parse_status",
            ]
        ]
        .drop_duplicates()
        .head(50)
    )


SALT PARSING STATUS

NaCl:
NaCl_parse_status
single       1764
absent        843
range         196
open_low       20
open_high      18
Name: count, dtype: int64

KCl:
KCl_parse_status
absent       2481
single        346
range          11
open_low        2
open_high       1
Name: count, dtype: int64

MgCl2:
MgCl2_parse_status
absent      2759
single        81
open_low       1
Name: count, dtype: int64

Representative salt examples:
    Salt concentration  NaCl_min_mM  NaCl_max_mM  KCl_min_mM  KCl_max_mM  \
0          125 mM NaCl        125.0        125.0         0.0         0.0   
2          150 mM NaCl        150.0        150.0         0.0         0.0   
14         500 mM NaCl        500.0        500.0         0.0         0.0   
15        1000 mM NaCl       1000.0       1000.0         0.0         0.0   
16          500 mM KCl          0.0          0.0       500.0       500.0   
..                 ...          ...          ...         ...         ...   
704     0-1000 mM NaBr          

In [10]:
llps[['Salt concentration', 'NaCl_min_mM',  'NaCl_max_mM' , 'KCl_min_mM' , 'KCl_max_mM' , 'MgCl2_min_mM' ,  'MgCl2_max_mM' ]].sample(50)

,Salt concentration,NaCl_min_mM,NaCl_max_mM,KCl_min_mM,KCl_max_mM,MgCl2_min_mM,MgCl2_max_mM
296,30-330 mM NaCl,30.0,330.0,0.0,0.0,0.0,0.0
958,150 mM NaCl,150.0,150.0,0.0,0.0,0.0,0.0
1338,100 mM NaCl;40 µM ZnSO4,100.0,100.0,0.0,0.0,0.0,0.0
110,50 mM NaCl,50.0,50.0,0.0,0.0,0.0,0.0
1297,2-100 μM NiCl2,0.0,0.0,0.0,0.0,0.0,0.0
217,150 mM KCl;2.5 mM MgCl2,0.0,0.0,150.0,150.0,2.5,2.5
554,200 mM NaCl,200.0,200.0,0.0,0.0,0.0,0.0
902,150 mM NaCl,150.0,150.0,0.0,0.0,0.0,0.0
2685,300-900 mM NaCl,300.0,900.0,0.0,0.0,0.0,0.0
2678,100mM NaCl,100.0,100.0,0.0,0.0,0.0,0.0


In [11]:
import re
import numpy as np
import pandas as pd


MAX_CAGENT_PCT = 50.0

CROWDING_CATEGORIES = [
    "PEG_300_1k",
    "PEG_3k_6k",
    "PEG_8k_20k",
    "Ficoll",
    "Dextran_le40k",
    "Dextran_ge70k",
]

UNIT_PATTERN = (
    r"(?:(?:wt|w\s*/\s*v|v\s*/\s*v)\s*%|%|"
    r"mg\s*/\s*ml|g\s*/\s*l)"
)


def normalize_crowding_text(text):
    if pd.isna(text):
        return None

    text = str(text).strip()

    if not text or text.upper() in {"-", "NA", "N/A", "NAN"}:
        return None

    text = (
        text.replace("μ", "u")
        .replace("µ", "u")
        .replace("–", "-")
        .replace("−", "-")
        .replace("＜", "<")
        .replace("＞", ">")
        .replace("≤", "<=")
        .replace("≥", ">=")
        .replace("·", "/")
    )

    text = re.sub(r"ml\s*-\s*1\b", "ml", text, flags=re.I)

    return re.sub(r"\s+", " ", text).strip()


def concentration_to_percent(value, unit):
    unit = unit.lower().replace(" ", "")

    if unit.endswith("%"):
        return float(value)

    if unit in {"mg/ml", "mg/ml−1", "mgml"}:
        return float(value) / 10.0

    if unit in {"g/l", "gl"}:
        return float(value) / 10.0

    return np.nan


def classify_crowding_agent(text):
    text = text.lower()

    # PEG
    if re.search(r"\bpeg|polyethylene glycol", text):
        mw_match = re.search(
            r"(?:peg|polyethylene glycol)[\s-]*"
            r"(\d+(?:\.\d+)?)\s*(k|kda)?",
            text,
            flags=re.I,
        )

        mw = None

        if mw_match:
            mw = float(mw_match.group(1))

            if mw_match.group(2):
                mw *= 1000.0

        if mw is None:
            return "PEG_3k_6k"

        if mw <= 1000:
            return "PEG_300_1k"

        if mw <= 6000:
            return "PEG_3k_6k"

        return "PEG_8k_20k"

    # Ficoll
    if "ficol" in text:
        return "Ficoll"

    # Dextran
    if "dextran" in text:
        range_match = re.search(
            r"dextran\w*[\s\-(]*"
            r"(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)\s*(k|kda|da)?",
            text,
            flags=re.I,
        )

        if range_match:
            low = float(range_match.group(1))
            high = float(range_match.group(2))
            unit = range_match.group(3)

            if unit in {"k", "kda"}:
                low *= 1000.0
                high *= 1000.0

            mw = (low + high) / 2.0

            if mw <= 40000:
                return "Dextran_le40k"

            return "Dextran_ge70k"

        single_match = re.search(
            r"dextran\w*[\s\-(]*t?[\s-]*"
            r"(\d+(?:\.\d+)?)\s*(k|kda|da)?",
            text,
            flags=re.I,
        )

        if single_match:
            value = float(single_match.group(1))
            unit = single_match.group(2)

            if unit in {"k", "kda"}:
                mw = value * 1000.0
            elif unit == "da":
                mw = value
            elif value <= 2000:
                mw = value * 1000.0
            else:
                mw = value

            if mw <= 40000:
                return "Dextran_le40k"

            return "Dextran_ge70k"

        return "Dextran_le40k"

    return None


def parse_crowding(text):
    result = {}

    for category in CROWDING_CATEGORIES:
        result[f"{category}_min_pct"] = 0.0
        result[f"{category}_max_pct"] = 0.0
        result[f"{category}_parse_status"] = "absent"

    text = normalize_crowding_text(text)

    if text is None:
        return result

    parts = re.split(r";|,", text)

    for part in parts:
        category = classify_crowding_agent(part)

        if category is None:
            continue

        # Concentration range
        match = re.search(
            r"(\d+(?:\.\d+)?)\s*%?\s*-\s*"
            r"(\d+(?:\.\d+)?)\s*"
            rf"({UNIT_PATTERN})",
            part,
            flags=re.I,
        )

        if match:
            value_a = concentration_to_percent(
                float(match.group(1)),
                match.group(3),
            )
            value_b = concentration_to_percent(
                float(match.group(2)),
                match.group(3),
            )

            if not np.isnan(value_a) and not np.isnan(value_b):
                low = min(value_a, value_b)
                high = max(value_a, value_b)

                result[f"{category}_min_pct"] = low
                result[f"{category}_max_pct"] = high
                result[f"{category}_parse_status"] = "range"

                continue

        # Open-ended upper range
        match = re.search(
            r"(>=|>|≥)\s*"
            r"(\d+(?:\.\d+)?)\s*"
            rf"({UNIT_PATTERN})",
            part,
            flags=re.I,
        )

        if match:
            value = concentration_to_percent(
                float(match.group(2)),
                match.group(3),
            )

            if not np.isnan(value):
                result[f"{category}_min_pct"] = value
                result[f"{category}_max_pct"] = min(
                    MAX_CAGENT_PCT,
                    value * 10.0,
                )
                result[f"{category}_parse_status"] = "open_high"

                continue

        # Open-ended lower range
        match = re.search(
            r"(<=|<|≤)\s*"
            r"(\d+(?:\.\d+)?)\s*"
            rf"({UNIT_PATTERN})",
            part,
            flags=re.I,
        )

        if match:
            value = concentration_to_percent(
                float(match.group(2)),
                match.group(3),
            )

            if not np.isnan(value):
                result[f"{category}_min_pct"] = max(
                    0.0,
                    value / 10.0,
                )
                result[f"{category}_max_pct"] = value
                result[f"{category}_parse_status"] = "open_low"

                continue

        # Single value
        match = re.search(
            r"(\d+(?:\.\d+)?)\s*"
            rf"({UNIT_PATTERN})",
            part,
            flags=re.I,
        )

        if match:
            value = concentration_to_percent(
                float(match.group(1)),
                match.group(2),
            )

            if not np.isnan(value):
                result[f"{category}_min_pct"] = value
                result[f"{category}_max_pct"] = value
                result[f"{category}_parse_status"] = "single"

    return result


crowding_parsed = (
    llps["Crowding agent"]
    .apply(parse_crowding)
    .apply(pd.Series)
)

existing_columns = [
    col for col in crowding_parsed.columns
    if col in llps.columns
]

llps = llps.drop(columns=existing_columns)
llps = pd.concat([llps, crowding_parsed], axis=1)


print("\n" + "=" * 70)
print("CROWDING-AGENT PARSING STATUS")
print("=" * 70)

for category in CROWDING_CATEGORIES:
    print(f"\n{category}:")
    print(
        llps[f"{category}_parse_status"]
        .value_counts(dropna=False)
    )


print("\nRepresentative crowding-agent examples:")
print(
    llps[
        [
            "Crowding agent",
            "PEG_300_1k_min_pct",
            "PEG_300_1k_max_pct",
            "PEG_3k_6k_min_pct",
            "PEG_3k_6k_max_pct",
            "PEG_8k_20k_min_pct",
            "PEG_8k_20k_max_pct",
            "Ficoll_min_pct",
            "Ficoll_max_pct",
            "Dextran_le40k_min_pct",
            "Dextran_le40k_max_pct",
            "Dextran_ge70k_min_pct",
            "Dextran_ge70k_max_pct",
        ]
    ]
    .drop_duplicates()
    .head(100)
)


known_patterns = re.compile(
    r"peg|polyethylene glycol|ficoll|dextran",
    flags=re.I,
)

crowding_present = (
    llps["Crowding agent"].notna()
    & ~llps["Crowding agent"]
    .astype(str)
    .str.strip()
    .isin(["", "-"])
)

unrecognized_crowding = llps[
    crowding_present
    & ~llps["Crowding agent"]
    .astype(str)
    .str.contains(known_patterns, na=False)
]

print("\nCrowding-agent entries outside LLPSense's six categories:")
print(
    unrecognized_crowding[["Crowding agent"]]
    .drop_duplicates()
    .head(100)
)


CROWDING-AGENT PARSING STATUS

PEG_300_1k:
PEG_300_1k_parse_status
absent       2718
single        107
range          12
open_high       4
Name: count, dtype: int64

PEG_3k_6k:
PEG_3k_6k_parse_status
absent       2546
single        279
range          14
open_high       1
open_low        1
Name: count, dtype: int64

PEG_8k_20k:
PEG_8k_20k_parse_status
absent    2565
single     227
range       49
Name: count, dtype: int64

Ficoll:
Ficoll_parse_status
absent    2759
single      79
range        3
Name: count, dtype: int64

Dextran_le40k:
Dextran_le40k_parse_status
absent    2710
single     128
range        3
Name: count, dtype: int64

Dextran_ge70k:
Dextran_ge70k_parse_status
absent    2810
single      30
range        1
Name: count, dtype: int64

Representative crowding-agent examples:
                            Crowding agent  PEG_300_1k_min_pct  \
0                                        -                 0.0   
7                     10 mg/ml dextrans-4K                 0.0   
8       

In [12]:
llps[['Crowding agent',   'PEG_300_1k_min_pct' , 'PEG_300_1k_max_pct' ,  'PEG_3k_6k_min_pct',   'PEG_3k_6k_max_pct',   'PEG_8k_20k_min_pct',  'PEG_8k_20k_max_pct',  'Ficoll_min_pct',  'Ficoll_max_pct', 'Dextran_le40k_min_pct',  'Dextran_le40k_max_pct',   'Dextran_ge70k_min_pct',  'Dextran_ge70k_max_pct'  ]].sample(50)

,Crowding agent,PEG_300_1k_min_pct,PEG_300_1k_max_pct,PEG_3k_6k_min_pct,PEG_3k_6k_max_pct,PEG_8k_20k_min_pct,PEG_8k_20k_max_pct,Ficoll_min_pct,Ficoll_max_pct,Dextran_le40k_min_pct,Dextran_le40k_max_pct,Dextran_ge70k_min_pct,Dextran_ge70k_max_pct
735,-,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
651,-,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
221,-,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
321,-,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2731,-,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1743,-,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2030,10% Ficoll-400,0.0,0.0,0.0,0.0,0.0,0.0,10.0,10.0,0.0,0.0,0.0,0.0
1607,-,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1565,-,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2670,10% Ficoll,0.0,0.0,0.0,0.0,0.0,0.0,10.0,10.0,0.0,0.0,0.0,0.0


In [13]:
MAX_GLYCEROL_PCT = 10.0

GLYCEROL_WORD_RE = re.compile(r"\bglycerol\b", re.IGNORECASE)


def normalize_glycerol_text(text):
    if pd.isna(text):
        return None

    text = str(text).strip()

    if not text or text.upper() in {"-", "NA", "N/A", "NAN"}:
        return None

    text = (
        text.replace("μ", "u")
        .replace("µ", "u")
        .replace("–", "-")
        .replace("−", "-")
        .replace("＜", "<")
        .replace("＞", ">")
        .replace("≤", "<=")
        .replace("≥", ">=")
    )

    return re.sub(r"\s+", " ", text).strip()


def parse_glycerol_from_text(text):
    text = normalize_glycerol_text(text)

    if text is None:
        return np.nan, np.nan, "missing"

    if not GLYCEROL_WORD_RE.search(text):
        return 0.0, 0.0, "absent"

    # Range
    match = re.search(
        r"(\d+(?:\.\d+)?)\s*%?\s*(?:-|to)\s*"
        r"(\d+(?:\.\d+)?)\s*%"
        r"(?:\s*\(?\s*[vw]\s*/\s*v\s*\)?)?"
        r"\s*\bglycerol\b",
        text,
        flags=re.I,
    )

    if match:
        value_a = float(match.group(1))
        value_b = float(match.group(2))

        return min(value_a, value_b), max(value_a, value_b), "range"

    # Open-ended lower bound
    match = re.search(
        r"(>=|>|≥)\s*"
        r"(\d+(?:\.\d+)?)\s*%"
        r"(?:\s*\(?\s*[vw]\s*/\s*v\s*\)?)?"
        r"\s*\bglycerol\b",
        text,
        flags=re.I,
    )

    if match:
        value = float(match.group(2))

        return (
            value,
            min(MAX_GLYCEROL_PCT, value * 10.0),
            "open_high",
        )

    # Open-ended upper bound
    match = re.search(
        r"(<=|<|≤)\s*"
        r"(\d+(?:\.\d+)?)\s*%"
        r"(?:\s*\(?\s*[vw]\s*/\s*v\s*\)?)?"
        r"\s*\bglycerol\b",
        text,
        flags=re.I,
    )

    if match:
        value = float(match.group(2))

        return (
            max(0.0, value / 10.0),
            value,
            "open_low",
        )

    # Single value
    match = re.search(
        r"(\d+(?:\.\d+)?)\s*%"
        r"(?:\s*\(?\s*[vw]\s*/\s*v\s*\)?)?"
        r"\s*\bglycerol\b",
        text,
        flags=re.I,
    )

    if match:
        value = float(match.group(1))
        return value, value, "single"

    return np.nan, np.nan, "unparsed"


def parse_glycerol(row):
    fields = [
        row["Buffer"],
        row["Crowding agent"],
        row["other molecules"],
    ]

    valid_statuses = {
        "single",
        "range",
        "open_high",
        "open_low",
    }

    for value in fields:
        result = parse_glycerol_from_text(value)

        if result[2] in valid_statuses:
            return result

    for value in fields:
        result = parse_glycerol_from_text(value)

        if result[2] == "unparsed":
            return result

    return 0.0, 0.0, "absent"


glycerol_parsed = (
    llps.apply(parse_glycerol, axis=1)
    .apply(pd.Series)
)

glycerol_parsed.columns = [
    "glycerol_min_pct",
    "glycerol_max_pct",
    "glycerol_parse_status",
]

existing_columns = [
    col for col in glycerol_parsed.columns
    if col in llps.columns
]

llps = llps.drop(columns=existing_columns)
llps = pd.concat([llps, glycerol_parsed], axis=1)


print("\n" + "=" * 70)
print("GLYCEROL PARSING STATUS")
print("=" * 70)

print(
    llps["glycerol_parse_status"]
    .value_counts(dropna=False)
)


print("\nRepresentative glycerol examples:")
print(
    llps[
        [
            "Buffer",
            "Crowding agent",
            "other molecules",
            "glycerol_min_pct",
            "glycerol_max_pct",
            "glycerol_parse_status",
        ]
    ]
    .drop_duplicates()
    .head(100)
)


print("\nUnparsed glycerol entries:")
print(
    llps[
        llps["glycerol_parse_status"] == "unparsed"
    ][
        [
            "Buffer",
            "Crowding agent",
            "other molecules",
            "glycerol_min_pct",
            "glycerol_max_pct",
            "glycerol_parse_status",
        ]
    ]
    .drop_duplicates()
    .head(100)
)


GLYCEROL PARSING STATUS
glycerol_parse_status
absent    2527
single     313
range        1
Name: count, dtype: int64

Representative glycerol examples:
                                                Buffer        Crowding agent  \
0                         20 mM Tris, pH 7.4, 1 mM DTT                     -   
1                         20 mM Tris, pH 7.4, 1 mM DTT                     -   
2                                   20 mM Tris, pH 7.5                     -   
6                                   20 mM Tris, pH 7.5                     -   
7                                   20 mM Tris, pH 7.5  10 mg/ml dextrans-4K   
..                                                 ...                   ...   
205                         20mM HEPES pH 7.2, 1mM DTT                     -   
209            20 mM NaPhosphate, 0.5 mM EDTA (pH 6.8)                     -   
210  pH 7.4 buffer consisting of 20 mM NaPhosphate,...      150 mg/ml Ficoll   
214                                 20 mM HEPES

In [14]:
required_columns = [
    "PSID",
    "Protein ID",
    "Sequence_clean",
    "llps_label",
    "temperature_min_C",
    "temperature_max_C",
    "protein_conc_min_uM",
    "protein_conc_max_uM",
    "pH_min",
    "pH_max",
    "PEG_300_1k_min_pct",
    "PEG_300_1k_max_pct",
    "PEG_3k_6k_min_pct",
    "PEG_3k_6k_max_pct",
    "PEG_8k_20k_min_pct",
    "PEG_8k_20k_max_pct",
    "Ficoll_min_pct",
    "Ficoll_max_pct",
    "Dextran_le40k_min_pct",
    "Dextran_le40k_max_pct",
    "Dextran_ge70k_min_pct",
    "Dextran_ge70k_max_pct",
    "MgCl2_min_mM",
    "MgCl2_max_mM",
    "NaCl_min_mM",
    "NaCl_max_mM",
    "KCl_min_mM",
    "KCl_max_mM",
    "glycerol_min_pct",
    "glycerol_max_pct",
]

missing_columns = [
    col for col in required_columns
    if col not in llps.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

llps_review = llps[required_columns].copy()

llps_review.rename(
    columns={
        "Sequence_clean": "Sequence",
        "llps_label": "Label",
    },
    inplace=True,
)

output_file = "LLPSDB_v2_LLPSense_intermediate.csv"

llps_review.to_csv(output_file, index=False)

print(
    f"Saved {len(llps_review):,} rows and "
    f"{len(llps_review.columns)} columns."
)

print("\nColumns:")
print(llps_review.columns.tolist())

Saved 2,841 rows and 30 columns.

Columns:
['PSID', 'Protein ID', 'Sequence', 'Label', 'temperature_min_C', 'temperature_max_C', 'protein_conc_min_uM', 'protein_conc_max_uM', 'pH_min', 'pH_max', 'PEG_300_1k_min_pct', 'PEG_300_1k_max_pct', 'PEG_3k_6k_min_pct', 'PEG_3k_6k_max_pct', 'PEG_8k_20k_min_pct', 'PEG_8k_20k_max_pct', 'Ficoll_min_pct', 'Ficoll_max_pct', 'Dextran_le40k_min_pct', 'Dextran_le40k_max_pct', 'Dextran_ge70k_min_pct', 'Dextran_ge70k_max_pct', 'MgCl2_min_mM', 'MgCl2_max_mM', 'NaCl_min_mM', 'NaCl_max_mM', 'KCl_min_mM', 'KCl_max_mM', 'glycerol_min_pct', 'glycerol_max_pct']


In [15]:
# now we will start sampleing as in LLPS SENSE

In [16]:
import itertools
import random
import numpy as np
import pandas as pd


GUIDE = {
    "temperature": [60.0, 50, 0.0, 60.0, 1],
    "protein_conc": [1000.0, 10, 0.0, 1000.0, 0],
    "pH": [14.0, 10, 0.0, 14.0, 0.1],
    "MgCl2": [50.0, 10, 0.0, 50.0, 1],
    "NaCl": [2000.0, 50, 0.0, 2000.0, 10],
    "KCl": [1000.0, 50, 0.0, 1000.0, 10],
    "crowding": [50.0, 30, 0.0, 50.0, 1],
    "glycerol": [10.0, 10, 0.0, 10.0, 1],
}


def sample_range(min_value, max_value, guide, parse_status):
    if pd.isna(min_value) or pd.isna(max_value):
        return None

    _, num_split, _, _, num_unit = guide

    if min_value == max_value:
        return [float(min_value)]

    is_open_range = parse_status in {"open_high", "open_low"}
    reverse_order = parse_status == "open_low"

    if num_unit == 0:
        step = (max_value - min_value) / num_split

        values = np.linspace(
            min_value,
            max_value - step,
            num_split,
        )
    else:
        values = np.arange(
            min_value,
            max_value - num_unit,
            num_unit,
        )

        if len(values) > num_split and is_open_range:
            values = (
                values[-num_split:]
                if reverse_order
                else values[:num_split]
            )

        elif not is_open_range and len(values) < 10:
            step = (max_value - min_value) / 10

            values = np.linspace(
                min_value,
                max_value - step,
                10,
            )

    return values.tolist()


CONDITIONS = {
    "temperature_C": (
        "temperature_min_C",
        "temperature_max_C",
        "temperature_parse_status",
        GUIDE["temperature"],
    ),
    "protein_conc_uM": (
        "protein_conc_min_uM",
        "protein_conc_max_uM",
        "protein_conc_parse_status",
        GUIDE["protein_conc"],
    ),
    "pH": (
        "pH_min",
        "pH_max",
        "pH_parse_status",
        GUIDE["pH"],
    ),
    "PEG_300_1k_pct": (
        "PEG_300_1k_min_pct",
        "PEG_300_1k_max_pct",
        "PEG_300_1k_parse_status",
        GUIDE["crowding"],
    ),
    "PEG_3k_6k_pct": (
        "PEG_3k_6k_min_pct",
        "PEG_3k_6k_max_pct",
        "PEG_3k_6k_parse_status",
        GUIDE["crowding"],
    ),
    "PEG_8k_20k_pct": (
        "PEG_8k_20k_min_pct",
        "PEG_8k_20k_max_pct",
        "PEG_8k_20k_parse_status",
        GUIDE["crowding"],
    ),
    "Ficoll_pct": (
        "Ficoll_min_pct",
        "Ficoll_max_pct",
        "Ficoll_parse_status",
        GUIDE["crowding"],
    ),
    "Dextran_le40k_pct": (
        "Dextran_le40k_min_pct",
        "Dextran_le40k_max_pct",
        "Dextran_le40k_parse_status",
        GUIDE["crowding"],
    ),
    "Dextran_ge70k_pct": (
        "Dextran_ge70k_min_pct",
        "Dextran_ge70k_max_pct",
        "Dextran_ge70k_parse_status",
        GUIDE["crowding"],
    ),
    "MgCl2_mM": (
        "MgCl2_min_mM",
        "MgCl2_max_mM",
        "MgCl2_parse_status",
        GUIDE["MgCl2"],
    ),
    "NaCl_mM": (
        "NaCl_min_mM",
        "NaCl_max_mM",
        "NaCl_parse_status",
        GUIDE["NaCl"],
    ),
    "KCl_mM": (
        "KCl_min_mM",
        "KCl_max_mM",
        "KCl_parse_status",
        GUIDE["KCl"],
    ),
    "glycerol_pct": (
        "glycerol_min_pct",
        "glycerol_max_pct",
        "glycerol_parse_status",
        GUIDE["glycerol"],
    ),
}


def expand_one_row(row):
    sampled_conditions = {}

    for name, (min_col, max_col, status_col, guide) in CONDITIONS.items():
        values = sample_range(
            row[min_col],
            row[max_col],
            guide,
            row[status_col],
        )

        if values is None:
            return []

        sampled_conditions[name] = values

    condition_names = list(sampled_conditions)

    combinations = list(
        itertools.product(
            *(sampled_conditions[name] for name in condition_names)
        )
    )

    if len(combinations) > 100:
        sample_size = max(100, len(combinations) // 10)
        combinations = random.sample(combinations, sample_size)

    expanded_rows = []

    for combination in combinations:
        new_row = {
            "PSID": row["PSID"],
            "Protein_ID": row["Protein ID"],
            "Sequence": row["Sequence_clean"],
            "Label": row["llps_label"],
        }

        new_row.update(zip(condition_names, combination))
        expanded_rows.append(new_row)

    return expanded_rows


random.seed(42)

expanded_data = []

for _, row in llps.iterrows():
    expanded_data.extend(expand_one_row(row))


llpsense_dataset = pd.DataFrame(expanded_data)


print("\n" + "=" * 70)
print("LLPSENSE SAMPLING / EXPANSION")
print("=" * 70)

print(f"Original experimental rows : {len(llps):,}")
print(f"Expanded rows              : {len(llpsense_dataset):,}")
print(
    f"Unique PSIDs               : "
    f"{llpsense_dataset['PSID'].nunique():,}"
)
print(
    f"Unique proteins            : "
    f"{llpsense_dataset['Protein_ID'].nunique():,}"
)

print("\nLabel distribution:")
print(llpsense_dataset["Label"].value_counts())

print("\nSamples generated per PSID:")
print(
    llpsense_dataset
    .groupby("PSID")
    .size()
    .describe()
)

print("\nFinal columns:")
print(llpsense_dataset.columns.tolist())


LLPSENSE SAMPLING / EXPANSION
Original experimental rows : 2,841
Expanded rows              : 20,719
Unique PSIDs               : 2,626
Unique proteins            : 249

Label distribution:
Label
1    11660
0     9059
Name: count, dtype: int64

Samples generated per PSID:
count    2626.000000
mean        7.889947
std        17.215267
min         1.000000
25%         1.000000
50%         1.000000
75%        10.000000
max       176.000000
dtype: float64

Final columns:
['PSID', 'Protein_ID', 'Sequence', 'Label', 'temperature_C', 'protein_conc_uM', 'pH', 'PEG_300_1k_pct', 'PEG_3k_6k_pct', 'PEG_8k_20k_pct', 'Ficoll_pct', 'Dextran_le40k_pct', 'Dextran_ge70k_pct', 'MgCl2_mM', 'NaCl_mM', 'KCl_mM', 'glycerol_pct']


In [17]:
LIMITS = {
    "temperature_C": 60.0,
    "protein_conc_uM": 1000.0,
    "pH": 14.0,
    "PEG_300_1k_pct": 50.0,
    "PEG_3k_6k_pct": 50.0,
    "PEG_8k_20k_pct": 50.0,
    "Ficoll_pct": 50.0,
    "Dextran_le40k_pct": 50.0,
    "Dextran_ge70k_pct": 50.0,
    "MgCl2_mM": 50.0,
    "NaCl_mM": 2000.0,
    "KCl_mM": 1000.0,
    "glycerol_pct": 10.0,
}


print("\n" + "=" * 70)
print("VALUES EXCEEDING LLPSENSE LIMITS")
print("=" * 70)

for column, limit in LIMITS.items():
    mask = llpsense_dataset[column] > limit

    if not mask.any():
        continue

    print(f"\n{column} (limit = {limit})")

    exceeding_values = (
        llpsense_dataset.loc[
            mask,
            ["PSID", "Protein_ID", column]
        ]
        .drop_duplicates()
        .sort_values(column, ascending=False)
        .head(30)
    )

    print(exceeding_values.to_string(index=False))
    print(f"Count: {mask.sum()}")


VALUES EXCEEDING LLPSENSE LIMITS

temperature_C (limit = 60.0)
      PSID Protein_ID  temperature_C
PS00004177      p0427           94.0
PS00001419      p0155           80.0
PS00001825      p0264           78.0
PS00001824      p0264           78.0
PS00001824      p0264           77.0
PS00001825      p0264           77.0
PS00001825      p0264           76.0
PS00001824      p0264           76.0
PS00001825      p0264           75.0
PS00001824      p0264           75.0
PS00001824      p0264           74.0
PS00001825      p0264           74.0
PS00001824      p0264           73.0
PS00002321      p0032           73.0
PS00001825      p0264           73.0
PS00001825      p0264           72.0
PS00002321      p0032           72.0
PS00001824      p0264           72.0
PS00001825      p0264           71.0
PS00002321      p0032           71.0
PS00001824      p0264           71.0
PS00001825      p0264           70.0
PS00002321      p0032           70.0
PS00001824      p0264           70.0
PS00001825 

In [18]:
MAX_VALUES = {
    "temperature_C": 60.0,
    "protein_conc_uM": 1000.0,
    "pH": 14.0,
    "PEG_300_1k_pct": 50.0,
    "PEG_3k_6k_pct": 50.0,
    "PEG_8k_20k_pct": 50.0,
    "Ficoll_pct": 50.0,
    "Dextran_le40k_pct": 50.0,
    "Dextran_ge70k_pct": 50.0,
    "MgCl2_mM": 50.0,
    "NaCl_mM": 2000.0,
    "KCl_mM": 1000.0,
    "glycerol_pct": 10.0,
}


llpsense_normalized = llpsense_dataset.copy()

for column, max_value in MAX_VALUES.items():
    llpsense_normalized[column] /= max_value


print("\n" + "=" * 70)
print("NORMALIZATION QC")
print("=" * 70)

for column in MAX_VALUES:
    print(
        f"{column:25s} "
        f"min={llpsense_normalized[column].min():.4f} "
        f"max={llpsense_normalized[column].max():.4f}"
    )


output_file = "LLPSDB_v2_LLPSense_sampled_normalized.csv"

llpsense_normalized.to_csv(output_file, index=False)

print(f"\nSaved: {output_file}")
print(f"Shape: {llpsense_normalized.shape}")


NORMALIZATION QC
temperature_C             min=-0.0333 max=1.5667
protein_conc_uM           min=0.0000 max=200.0000
pH                        min=0.2143 max=0.7143
PEG_300_1k_pct            min=0.0000 max=0.7800
PEG_3k_6k_pct             min=0.0000 max=0.7800
PEG_8k_20k_pct            min=0.0000 max=1.0000
Ficoll_pct                min=0.0000 max=0.7600
Dextran_le40k_pct         min=0.0000 max=0.3200
Dextran_ge70k_pct         min=0.0000 max=0.4000
MgCl2_mM                  min=0.0000 max=2.9600
NaCl_mM                   min=0.0000 max=75.0000
KCl_mM                    min=0.0000 max=2.7000
glycerol_pct              min=0.0000 max=3.8000

Saved: LLPSDB_v2_LLPSense_sampled_normalized.csv
Shape: (20719, 17)


In [19]:
# Install MMseqs2
!sudo apt-get update -qq
!sudo apt-get install -y mmseqs2

# Check installation
!mmseqs version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libgzstream0
The following NEW packages will be installed:
  libgzstream0 mmseqs2
0 upgraded, 2 newly installed, 0 to remove and 15 not upgraded.
Need to get 25.2 MB of archives.
After this operation, 72.0 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/universe amd64 libgzstream0 amd64 1.5+git20171107.9a20658-1 [7,410 B]
Get:2 http://archive.ubuntu.com/ubuntu noble/universe amd64 mmseqs2 amd64 15-6f452+ds-2 [25.1 MB]
Fetched 25.2 MB in 2s (14.0 MB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl

In [20]:


# Get unique sequences from the sampled dataset
unique_sequences = (
    llpsense_dataset[["Protein_ID", "Sequence"]]
    .drop_duplicates(subset=["Sequence"])
    .reset_index(drop=True)
)

# Create a stable sequence ID
unique_sequences["Sequence_ID"] = [
    f"SEQ_{i:06d}"
    for i in range(len(unique_sequences))
]

# Create FASTA file
with open("llpsense_unique_sequences.fasta", "w") as f:

    for _, row in unique_sequences.iterrows():

        f.write(f">{row['Sequence_ID']}\n")
        f.write(f"{row['Sequence']}\n")


print(f"Unique sequences: {len(unique_sequences):,}")
print("FASTA file created: llpsense_unique_sequences.fasta")

Unique sequences: 815
FASTA file created: llpsense_unique_sequences.fasta


In [21]:


!rm -rf mmseqs_db mmseqs_cluster mmseqs_tmp

!mmseqs easy-cluster \
    llpsense_unique_sequences.fasta \
    mmseqs_cluster \
    mmseqs_tmp \
    --min-seq-id 0.20 \
    -c 0.90 \
    --cov-mode 1

print("MMseqs2 clustering completed.")

Create directory mmseqs_tmp
easy-cluster llpsense_unique_sequences.fasta mmseqs_cluster mmseqs_tmp --min-seq-id 0.20 -c 0.90 --cov-mode 1 

MMseqs Version:                     	15-6f452+ds-2
Substitution matrix                 	aa:blosum62.out,nucl:nucleotide.out
Seed substitution matrix            	aa:VTML80.out,nucl:nucleotide.out
Sensitivity                         	4
k-mer length                        	0
Target search mode                  	0
k-score                             	seq:2147483647,prof:2147483647
Alphabet size                       	aa:21,nucl:5
Max sequence length                 	65535
Max results per query               	20
Split database                      	0
Split mode                          	2
Split memory limit                  	0
Coverage threshold                  	0.9
Coverage mode                       	1
Compositional bias                  	1
Compositional bias                  	1
Diagonal scoring                    	true
Exact k-mer matching          

In [22]:
clusters = pd.read_csv(
    "mmseqs_cluster_cluster.tsv",
    sep="\t",
    header=None,
    names=["representative", "member"]
)

print("Unique sequences:", clusters["member"].nunique())
print("Number of clusters:", clusters["representative"].nunique())

cluster_sizes = clusters.groupby("representative").size()

print("\nCluster size statistics:")
print(cluster_sizes.describe())

print("\nSingleton clusters:", (cluster_sizes == 1).sum())

Unique sequences: 815
Number of clusters: 191

Cluster size statistics:
count    191.000000
mean       4.267016
std        7.730794
min        1.000000
25%        1.000000
50%        1.000000
75%        4.000000
max       54.000000
dtype: float64

Singleton clusters: 98


In [23]:
clusters = pd.read_csv(
    "mmseqs_cluster_cluster.tsv",
    sep="\t",
    header=None,
    names=["representative", "member"],
)

seq_to_cluster = dict(
    zip(
        clusters["member"],
        clusters["representative"],
    )
)

unique_sequences["Cluster_ID"] = (
    unique_sequences["Sequence_ID"].map(seq_to_cluster)
)

missing_clusters = unique_sequences["Cluster_ID"].isna().sum()

print(f"Unique sequences without cluster: {missing_clusters}")

if missing_clusters > 0:
    raise ValueError("Some sequences did not receive a cluster ID.")


sequence_to_cluster = dict(
    zip(
        unique_sequences["Sequence"],
        unique_sequences["Cluster_ID"],
    )
)

llpsense_dataset["Cluster_ID"] = (
    llpsense_dataset["Sequence"].map(sequence_to_cluster)
)


print(
    f"\nRows without Cluster_ID: "
    f"{llpsense_dataset['Cluster_ID'].isna().sum()}"
)

print(
    f"Unique clusters in final dataset: "
    f"{llpsense_dataset['Cluster_ID'].nunique()}"
)

print("\nCluster size distribution in final dataset:")
print(
    llpsense_dataset
    .groupby("Cluster_ID")
    .size()
    .describe()
)


output_file = "LLPSDB_v2_LLPSense_sampled_with_clusters.csv"

llpsense_dataset.to_csv(output_file, index=False)

print(f"\nSaved: {output_file}")

Unique sequences without cluster: 0

Rows without Cluster_ID: 0
Unique clusters in final dataset: 191

Cluster size distribution in final dataset:
count     191.000000
mean      108.476440
std       308.866669
min         1.000000
25%         2.000000
50%        12.000000
75%        59.500000
max      2703.000000
dtype: float64

Saved: LLPSDB_v2_LLPSense_sampled_with_clusters.csv


In [24]:
!pip install BioPython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 29.1 MB/s eta 0:00:00


In [25]:

import sys
!conda install --yes --prefix {sys.prefix} -c bioconda ncbi-blast
!apt-get update
!apt-get install -y ncbi-blast+

/bin/bash: line 1: conda: command not found
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:3 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Fetched 3,917 B in 1s (4,576 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libmbedtls14t64 libmbedx509

In [26]:
# add the data.zip , iupred3.py, iupred3_lib.py

In [27]:
import zipfile
with zipfile.ZipFile('/content/data.zip', 'r') as z:
      z.extractall('/content/')

In [28]:
import os
import re
import math
import shutil
import subprocess

import numpy as np
import pandas as pd

from Bio.SeqUtils.ProtParam import ProteinAnalysis


try:
    import iupred3_lib
except ImportError:
    raise ImportError(
        "iupred3_lib is not available. Install/download the official "
        "IUPred3 standalone package and make sure iupred3_lib.py "
        "is on the Python path."
    )


if shutil.which("segmasker") is None:
    raise RuntimeError(
        "segmasker was not found. In Colab, run:\n"
        "!sudo apt-get update -qq\n"
        "!sudo apt-get install -y ncbi-blast+"
    )


AA_LIST = list("ACDEFGHIKLMNPQRSTVWY")

HYDROPHOBIC_AAS = set("AILMFV")
POLAR_AAS = set("SQNGCTP")
CATIONIC_AAS = set("KRH")
ANIONIC_AAS = set("DE")
AROMATIC_AAS = set("WYF")


KD = {
    "A": 1.8,
    "R": -4.5,
    "N": -3.5,
    "D": -3.5,
    "C": 2.5,
    "Q": -3.5,
    "E": -3.5,
    "G": -0.4,
    "H": -3.2,
    "I": 4.5,
    "L": 3.8,
    "K": -3.9,
    "M": 1.9,
    "F": 2.8,
    "P": -1.6,
    "S": -0.8,
    "T": -0.7,
    "W": -0.9,
    "Y": -1.3,
    "V": 4.2,
}


def validate_sequence(seq):
    if not isinstance(seq, str):
        return False

    seq = seq.upper().strip()

    return bool(seq) and set(seq).issubset(set(AA_LIST))


def calculate_basic_features(seq):
    analysis = ProteinAnalysis(seq)
    length = len(seq)

    molecular_weight = analysis.molecular_weight()
    pI = analysis.isoelectric_point()

    kd_values = np.array([KD[aa] for aa in seq])
    mean_hydrophobicity = kd_values.mean()

    counts = np.array(
        [seq.count(aa) for aa in AA_LIST],
        dtype=float,
    )

    probabilities = counts / length

    shannon_entropy = -sum(
        p * math.log2(p)
        for p in probabilities
        if p > 0
    )

    positive_count = sum(
        seq.count(aa)
        for aa in CATIONIC_AAS
    )

    negative_count = sum(
        seq.count(aa)
        for aa in ANIONIC_AAS
    )

    net_charge = positive_count - negative_count

    return {
        "Sequence_length": length,
        "Molecular_weight": molecular_weight,
        "pI": pI,
        "Mean_KD_hydrophobicity": mean_hydrophobicity,
        "Shannon_entropy": shannon_entropy,
        "Positive_charge_count": positive_count,
        "Negative_charge_count": negative_count,
        "Net_charge": net_charge,
        "Positive_charge_frac": positive_count / length,
        "Negative_charge_frac": negative_count / length,
        "Net_charge_per_residue": net_charge / length,
    }


def calculate_aa_features(seq):
    length = len(seq)
    features = {}

    for aa in AA_LIST:
        features[f"AA_{aa}_frac"] = seq.count(aa) / length

    residue_groups = {
        "HB": HYDROPHOBIC_AAS,
        "Polar": POLAR_AAS,
        "Arom": AROMATIC_AAS,
        "Cation": CATIONIC_AAS,
        "Anion": ANIONIC_AAS,
    }

    for group_name, residues in residue_groups.items():
        count = sum(seq.count(aa) for aa in residues)
        features[f"{group_name}_frac"] = count / length

    return features


def calculate_iupred3_features(seq):
    smoothing = "medium" if len(seq) >= 19 else "no"

    result = iupred3_lib.iupred(
        seq,
        mode="long",
        smoothing=smoothing,
    )

    scores = np.asarray(result[0], dtype=float)
    scores = scores[np.isfinite(scores)]

    if len(scores) == 0:
        return {
            "IUPred3_mean": np.nan,
            "IUPred3_median": np.nan,
            "IUPred3_max": np.nan,
            "IUPred3_disordered_frac": np.nan,
        }

    threshold = 0.5

    return {
        "IUPred3_mean": scores.mean(),
        "IUPred3_median": np.median(scores),
        "IUPred3_max": scores.max(),
        "IUPred3_disordered_frac": (scores > threshold).mean(),
    }


def run_segmasker(sequences):
    fasta_file = "unique_sequences_for_SEG.fasta"
    interval_file = "seg_intervals.txt"

    with open(fasta_file, "w") as file:
        for sequence_id, seq in sequences.items():
            file.write(f">{sequence_id}\n")
            file.write(f"{seq}\n")

    result = subprocess.run(
        [
            "segmasker",
            "-in",
            fasta_file,
            "-infmt",
            "fasta",
            "-outfmt",
            "interval",
            "-parse_seqids",
            "-out",
            interval_file,
        ],
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        raise RuntimeError(
            "segmasker failed:\n" + result.stderr
        )

    lcr_intervals = {
        sequence_id: []
        for sequence_id in sequences
    }

    current_id = None

    with open(interval_file, "r") as file:
        for line in file:
            line = line.strip()

            if not line:
                continue

            if line.startswith(">"):
                current_id = line[1:].strip()

                if current_id not in lcr_intervals:
                    lcr_intervals[current_id] = []

                continue

            match = re.match(r"(\d+)\s*-\s*(\d+)", line)

            if match and current_id is not None:
                start = int(match.group(1))
                end = int(match.group(2))
                lcr_intervals[current_id].append((start, end))

    for path in (fasta_file, interval_file):
        if os.path.exists(path):
            os.remove(path)

    return lcr_intervals


def calculate_lcr_features(seq, intervals):
    if not intervals:
        return {
            "LCR_length": 0,
            "LCR_frac": 0.0,
            "LCR_count": 0,
        }

    merged_intervals = []

    for start, end in sorted(intervals):
        if not merged_intervals:
            merged_intervals.append([start, end])

        elif start <= merged_intervals[-1][1] + 1:
            merged_intervals[-1][1] = max(
                merged_intervals[-1][1],
                end,
            )

        else:
            merged_intervals.append([start, end])

    lcr_length = sum(
        end - start + 1
        for start, end in merged_intervals
    )

    return {
        "LCR_length": lcr_length,
        "LCR_frac": lcr_length / len(seq),
        "LCR_count": len(merged_intervals),
    }


unique_sequences = (
    llpsense_dataset[["Sequence"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

unique_sequences["Sequence_ID"] = [
    f"SEQ_{i:06d}"
    for i in range(len(unique_sequences))
]

print(
    f"Unique sequences to featurize: "
    f"{len(unique_sequences):,}"
)


sequence_dict = dict(
    zip(
        unique_sequences["Sequence_ID"],
        unique_sequences["Sequence"],
    )
)


print("\nRunning SEG/segmasker...")
lcr_intervals = run_segmasker(sequence_dict)
print("SEG completed.")


feature_rows = []

for idx, row in unique_sequences.iterrows():
    seq = row["Sequence"]
    sequence_id = row["Sequence_ID"]

    if not validate_sequence(seq):
        raise ValueError(f"Invalid sequence for {sequence_id}")

    features = {
        "Sequence_ID": sequence_id,
        "Sequence": seq,
    }

    features.update(calculate_basic_features(seq))
    features.update(calculate_aa_features(seq))
    features.update(calculate_iupred3_features(seq))
    features.update(
        calculate_lcr_features(
            seq,
            lcr_intervals.get(sequence_id, []),
        )
    )

    feature_rows.append(features)

    if (idx + 1) % 25 == 0:
        print(
            f"Processed {idx + 1}/{len(unique_sequences)}"
        )


sequence_features = pd.DataFrame(feature_rows)


print("\n" + "=" * 70)
print("SEQUENCE FEATURE EXTRACTION COMPLETE")
print("=" * 70)

print("Feature table shape:", sequence_features.shape)

print(
    "Number of unique sequences:",
    sequence_features["Sequence_ID"].nunique(),
)

print("\nMissing values per feature:")

missing = (
    sequence_features.isna()
    .sum()
    .sort_values(ascending=False)
)

print(missing[missing > 0])


output_file = "LLPSDB_v2_sequence_features.csv"

sequence_features.to_csv(
    output_file,
    index=False,
)

print(f"\nSaved: {output_file}")


preview_columns = [
    "Sequence_ID",
    "Sequence_length",
    "Molecular_weight",
    "pI",
    "Mean_KD_hydrophobicity",
    "Shannon_entropy",
    "IUPred3_mean",
    "IUPred3_disordered_frac",
    "LCR_length",
    "LCR_frac",
    "HB_frac",
    "Polar_frac",
    "Arom_frac",
    "Cation_frac",
    "Anion_frac",
]

print("\nFeature preview:")
print(sequence_features[preview_columns].head())

Unique sequences to featurize: 815

Running SEG/segmasker...
SEG completed.
Processed 25/815
Processed 50/815
Processed 75/815
Processed 100/815
Processed 125/815
Processed 150/815
Processed 175/815
Processed 200/815
Processed 225/815
Processed 250/815
Processed 275/815
Processed 300/815
Processed 325/815
Processed 350/815
Processed 375/815
Processed 400/815
Processed 425/815
Processed 450/815
Processed 475/815
Processed 500/815
Processed 525/815
Processed 550/815
Processed 575/815
Processed 600/815
Processed 625/815
Processed 650/815
Processed 675/815
Processed 700/815
Processed 725/815
Processed 750/815
Processed 775/815
Processed 800/815

SEQUENCE FEATURE EXTRACTION COMPLETE
Feature table shape: (815, 45)
Number of unique sequences: 815

Missing values per feature:
Series([], dtype: int64)

Saved: LLPSDB_v2_sequence_features.csv

Feature preview:
  Sequence_ID  Sequence_length  Molecular_weight        pI  \
0  SEQ_000000              708        76341.6577  6.603747   
1  SEQ_000001 

In [29]:
print("\nAll feature columns:")
for i, col in enumerate(sequence_features.columns):
    print(i, col)

print("\nNumber of columns:", len(sequence_features.columns))

print("\nDuplicate column names:")
print(
    sequence_features.columns[
        sequence_features.columns.duplicated()
    ].tolist()
)


All feature columns:
0 Sequence_ID
1 Sequence
2 Sequence_length
3 Molecular_weight
4 pI
5 Mean_KD_hydrophobicity
6 Shannon_entropy
7 Positive_charge_count
8 Negative_charge_count
9 Net_charge
10 Positive_charge_frac
11 Negative_charge_frac
12 Net_charge_per_residue
13 AA_A_frac
14 AA_C_frac
15 AA_D_frac
16 AA_E_frac
17 AA_F_frac
18 AA_G_frac
19 AA_H_frac
20 AA_I_frac
21 AA_K_frac
22 AA_L_frac
23 AA_M_frac
24 AA_N_frac
25 AA_P_frac
26 AA_Q_frac
27 AA_R_frac
28 AA_S_frac
29 AA_T_frac
30 AA_V_frac
31 AA_W_frac
32 AA_Y_frac
33 HB_frac
34 Polar_frac
35 Arom_frac
36 Cation_frac
37 Anion_frac
38 IUPred3_mean
39 IUPred3_median
40 IUPred3_max
41 IUPred3_disordered_frac
42 LCR_length
43 LCR_frac
44 LCR_count

Number of columns: 45

Duplicate column names:
[]


In [30]:
print("\nLCR statistics:")
print(
    sequence_features[
        ["LCR_length", "LCR_frac", "LCR_count"]
    ].describe()
)

print(
    "\nSequences with LCR:",
    (sequence_features["LCR_length"] > 0).sum()
)

print(
    "Sequences without LCR:",
    (sequence_features["LCR_length"] == 0).sum()
)


LCR statistics:
       LCR_length  LCR_frac  LCR_count
count       815.0     815.0      815.0
mean          0.0       0.0        0.0
std           0.0       0.0        0.0
min           0.0       0.0        0.0
25%           0.0       0.0        0.0
50%           0.0       0.0        0.0
75%           0.0       0.0        0.0
max           0.0       0.0        0.0

Sequences with LCR: 0
Sequences without LCR: 815


In [31]:
print("\nIUPred3 statistics:")
print(
    sequence_features[
        [
            "IUPred3_mean",
            "IUPred3_median",
            "IUPred3_max",
            "IUPred3_disordered_frac"
        ]
    ].describe()
)


IUPred3 statistics:
       IUPred3_mean  IUPred3_median  IUPred3_max  IUPred3_disordered_frac
count    815.000000      815.000000   815.000000               815.000000
mean       0.583862        0.585078     0.882697                 0.626910
std        0.190608        0.214922     0.148071                 0.315864
min        0.023420        0.009097     0.032489                 0.000000
25%        0.457831        0.430774     0.848102                 0.378661
50%        0.616464        0.589850     0.940565                 0.680000
75%        0.739943        0.769626     0.971638                 0.927445
max        0.976553        0.992343     0.999872                 1.000000


In [32]:
import subprocess
import tempfile
import os

# Test on the first sequence
test_seq = sequence_features.loc[0, "Sequence"]

with tempfile.TemporaryDirectory() as tmp:
    fasta_file = os.path.join(tmp, "test.fasta")
    output_file = os.path.join(tmp, "test_output")

    with open(fasta_file, "w") as f:
        f.write(">test_sequence\n")
        f.write(test_seq + "\n")

    cmd = [
        "segmasker",
        "-in", fasta_file,
        "-out", output_file
    ]

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )

    print("Return code:", result.returncode)
    print("\nSTDOUT:")
    print(result.stdout[:2000])

    print("\nSTDERR:")
    print(result.stderr[:2000])

    print("\nOutput files:")
    print(os.listdir(tmp))

    if os.path.exists(output_file):
        print("\nRAW OUTPUT:")
        with open(output_file, "r") as f:
            print(f.read()[:3000])

Return code: 0

STDOUT:


STDERR:


Output files:
['test_output', 'test.fasta']

RAW OUTPUT:
>test_sequence
2 - 20
27 - 94
96 - 131
623 - 649
654 - 697



In [33]:
import re

raw_output = """2 - 20
27 - 94
96 - 131
623 - 649
654 - 697"""

intervals = []

for line in raw_output.splitlines():
    match = re.match(r"^\s*(\d+)\s*-\s*(\d+)\s*$", line)

    if match:
        start = int(match.group(1))
        end = int(match.group(2))
        intervals.append((start, end))

print("Intervals:", intervals)

lcr_length = sum(end - start + 1 for start, end in intervals)
lcr_count = len(intervals)

print("LCR length:", lcr_length)
print("LCR count:", lcr_count)

Intervals: [(2, 20), (27, 94), (96, 131), (623, 649), (654, 697)]
LCR length: 194
LCR count: 5


In [34]:
import os
import re
import tempfile
import subprocess
import pandas as pd

# Use the existing sequence_features table
seq_df = sequence_features[["Sequence_ID", "Sequence"]].copy()

with tempfile.TemporaryDirectory() as tmp:

    fasta_file = os.path.join(tmp, "sequences.fasta")
    output_file = os.path.join(tmp, "seg_output")

    # Write all sequences
    with open(fasta_file, "w") as f:
        for _, row in seq_df.iterrows():
            f.write(f">{row['Sequence_ID']}\n")
            f.write(f"{row['Sequence']}\n")

    # Run SEG
    result = subprocess.run(
        ["segmasker", "-in", fasta_file, "-out", output_file],
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        raise RuntimeError(result.stderr)

    # Parse SEG output
    lcr_features = {}

    current_id = None
    current_intervals = []

    with open(output_file, "r") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            # New sequence header
            if line.startswith(">"):
                # Save previous sequence
                if current_id is not None:
                    seq_len = len(
                        seq_df.loc[
                            seq_df["Sequence_ID"] == current_id, "Sequence"
                        ].iloc[0]
                    )

                    lcr_length = sum(
                        end - start + 1
                        for start, end in current_intervals
                    )

                    lcr_count = len(current_intervals)

                    lcr_features[current_id] = {
                        "LCR_length": lcr_length,
                        "LCR_frac": lcr_length / seq_len if seq_len > 0 else 0,
                        "LCR_count": lcr_count
                    }

                current_id = line[1:].strip()
                current_intervals = []

            # LCR interval
            else:
                match = re.match(r"^\s*(\d+)\s*-\s*(\d+)\s*$", line)

                if match:
                    start = int(match.group(1))
                    end = int(match.group(2))
                    current_intervals.append((start, end))

        # Save last sequence
        if current_id is not None:
            seq_len = len(
                seq_df.loc[
                    seq_df["Sequence_ID"] == current_id, "Sequence"
                ].iloc[0]
            )

            lcr_length = sum(
                end - start + 1
                for start, end in current_intervals
            )

            lcr_count = len(current_intervals)

            lcr_features[current_id] = {
                "LCR_length": lcr_length,
                "LCR_frac": lcr_length / seq_len if seq_len > 0 else 0,
                "LCR_count": lcr_count
            }


# Convert to DataFrame
lcr_df = pd.DataFrame.from_dict(
    lcr_features,
    orient="index"
).reset_index().rename(columns={"index": "Sequence_ID"})

# Merge into existing feature table
sequence_features = sequence_features.drop(
    columns=["LCR_length", "LCR_frac", "LCR_count"]
)

sequence_features = sequence_features.merge(
    lcr_df,
    on="Sequence_ID",
    how="left"
)

sequence_features[["LCR_length", "LCR_frac", "LCR_count"]] = (
    sequence_features[["LCR_length", "LCR_frac", "LCR_count"]]
    .fillna(0)
)

print(sequence_features[["LCR_length", "LCR_frac", "LCR_count"]].describe())

print(
    "\nSequences with LCR:",
    (sequence_features["LCR_length"] > 0).sum()
)

print(
    "Sequences without LCR:",
    (sequence_features["LCR_length"] == 0).sum()
)

       LCR_length    LCR_frac   LCR_count
count  815.000000  815.000000  815.000000
mean   100.958282    0.332622    3.985276
std    102.101581    0.279047    3.333219
min      0.000000    0.000000    0.000000
25%     30.000000    0.112135    2.000000
50%     75.000000    0.279528    3.000000
75%    135.000000    0.455502    6.000000
max    724.000000    1.109890   27.000000

Sequences with LCR: 728
Sequences without LCR: 87


In [35]:
row = sequence_features.loc[
    sequence_features["LCR_frac"].idxmax()
]

print("Sequence_ID:", row["Sequence_ID"])
print("Sequence length:", row["Sequence_length"])
print("LCR length:", row["LCR_length"])
print("LCR fraction:", row["LCR_frac"])
print("LCR count:", row["LCR_count"])

Sequence_ID: SEQ_000277
Sequence length: 91
LCR length: 101
LCR fraction: 1.10989010989011
LCR count: 3


In [36]:
import os
import tempfile
import subprocess

row = sequence_features.loc[
    sequence_features["Sequence_ID"] == "SEQ_000277"
].iloc[0]

seq = row["Sequence"]

print("Sequence length:", len(seq))
print("Sequence:", seq)

with tempfile.TemporaryDirectory() as tmp:
    fasta_file = os.path.join(tmp, "test.fasta")
    output_file = os.path.join(tmp, "test_output")

    with open(fasta_file, "w") as f:
        f.write(">SEQ_000277\n")
        f.write(seq + "\n")

    subprocess.run(
        ["segmasker", "-in", fasta_file, "-out", output_file],
        check=True,
        capture_output=True,
        text=True
    )

    print("\nSEG output:")
    with open(output_file, "r") as f:
        print(f.read())

Sequence length: 91
Sequence: AEPSEAGDAPPDDGPPGELGPDGPDGPEEKGRDRDRERRRSHRSERERRRDRDRDRDRDREHKRGERGSERGRDEARGGGGGQDNGLEGLG

SEG output:
>SEQ_000277
0 - 28
27 - 66
59 - 90



In [37]:
intervals = [(0, 28), (27, 66), (59, 90)]

intervals = sorted(intervals)

merged = []

for start, end in intervals:
    if not merged or start > merged[-1][1] + 1:
        merged.append([start, end])
    else:
        merged[-1][1] = max(merged[-1][1], end)

lcr_length = sum(end - start + 1 for start, end in merged)

print("Original intervals:", intervals)
print("Merged intervals:", merged)
print("Correct LCR length:", lcr_length)
print("LCR fraction:", lcr_length / 91)
print("LCR count:", len(merged))

Original intervals: [(0, 28), (27, 66), (59, 90)]
Merged intervals: [[0, 90]]
Correct LCR length: 91
LCR fraction: 1.0
LCR count: 1


In [38]:
import os
import re
import tempfile
import subprocess
import pandas as pd

seq_df = sequence_features[["Sequence_ID", "Sequence"]].copy()

lcr_features = {}

with tempfile.TemporaryDirectory() as tmp:

    fasta_file = os.path.join(tmp, "sequences.fasta")
    output_file = os.path.join(tmp, "seg_output")

    with open(fasta_file, "w") as f:
        for _, row in seq_df.iterrows():
            f.write(f">{row['Sequence_ID']}\n")
            f.write(f"{row['Sequence']}\n")

    result = subprocess.run(
        ["segmasker", "-in", fasta_file, "-out", output_file],
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        raise RuntimeError(result.stderr)

    current_id = None
    current_intervals = []

    def save_lcr(seq_id, intervals):
        seq_len = len(
            seq_df.loc[
                seq_df["Sequence_ID"] == seq_id, "Sequence"
            ].iloc[0]
        )

        # Merge overlapping/adjacent intervals
        intervals = sorted(intervals)
        merged = []

        for start, end in intervals:
            if not merged or start > merged[-1][1] + 1:
                merged.append([start, end])
            else:
                merged[-1][1] = max(merged[-1][1], end)

        lcr_length = sum(
            end - start + 1
            for start, end in merged
        )

        lcr_features[seq_id] = {
            "LCR_length": lcr_length,
            "LCR_frac": lcr_length / seq_len if seq_len > 0 else 0,
            "LCR_count": len(merged)
        }

    with open(output_file, "r") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            if line.startswith(">"):
                if current_id is not None:
                    save_lcr(current_id, current_intervals)

                current_id = line[1:].strip()
                current_intervals = []

            else:
                match = re.match(
                    r"^\s*(\d+)\s*-\s*(\d+)\s*$",
                    line
                )

                if match:
                    current_intervals.append(
                        (
                            int(match.group(1)),
                            int(match.group(2))
                        )
                    )

        if current_id is not None:
            save_lcr(current_id, current_intervals)


# Replace old LCR features
sequence_features = sequence_features.drop(
    columns=["LCR_length", "LCR_frac", "LCR_count"]
)

lcr_df = pd.DataFrame.from_dict(
    lcr_features,
    orient="index"
).reset_index().rename(
    columns={"index": "Sequence_ID"}
)

sequence_features = sequence_features.merge(
    lcr_df,
    on="Sequence_ID",
    how="left"
)

sequence_features[
    ["LCR_length", "LCR_frac", "LCR_count"]
] = sequence_features[
    ["LCR_length", "LCR_frac", "LCR_count"]
].fillna(0)

print(sequence_features[
    ["LCR_length", "LCR_frac", "LCR_count"]
].describe())

print(
    "\nLCR fraction > 1:",
    (sequence_features["LCR_frac"] > 1).sum()
)

print(
    "Sequences with LCR:",
    (sequence_features["LCR_length"] > 0).sum()
)

print(
    "Sequences without LCR:",
    (sequence_features["LCR_length"] == 0).sum()
)

       LCR_length    LCR_frac   LCR_count
count  815.000000  815.000000  815.000000
mean    95.732515    0.316422    2.970552
std     95.292358    0.261331    2.353129
min      0.000000    0.000000    0.000000
25%     30.000000    0.110489    1.000000
50%     72.000000    0.257426    2.000000
75%    127.500000    0.440096    4.000000
max    665.000000    1.000000   21.000000

LCR fraction > 1: 0
Sequences with LCR: 728
Sequences without LCR: 87


In [39]:
sequence_features.to_csv(
    "LLPSDB_v2_sequence_features.csv",
    index=False
)

print("Saved:", "LLPSDB_v2_sequence_features.csv")
print("Shape:", sequence_features.shape)

Saved: LLPSDB_v2_sequence_features.csv
Shape: (815, 45)


In [40]:
# Use the normalized dataset that already contains:
# environmental features + Label + Cluster_ID
model_dataset = llpsense_normalized.copy()

# Remove Sequence_ID/Sequence from the feature table only where necessary
# Sequence is used as the matching key.
sequence_feature_columns = [
    col for col in sequence_features.columns
    if col not in ["Sequence_ID"]
]

# Merge sequence features using the experimental sequence
model_dataset = model_dataset.merge(
    sequence_features[sequence_feature_columns],
    on="Sequence",
    how="left",
    validate="many_to_one"
)

print("Model dataset shape:", model_dataset.shape)

print("\nMissing sequence features:")
print(
    model_dataset[sequence_feature_columns.drop if False else
                  [c for c in sequence_feature_columns if c != "Sequence"]]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print("\nColumns:", len(model_dataset.columns))

Model dataset shape: (20719, 60)

Missing sequence features:
Sequence_length           0
Molecular_weight          0
pI                        0
Mean_KD_hydrophobicity    0
Shannon_entropy           0
Positive_charge_count     0
Negative_charge_count     0
Net_charge                0
Positive_charge_frac      0
Negative_charge_frac      0
dtype: int64

Columns: 60


In [41]:
sequence_feature_cols = [
    col for col in sequence_features.columns
    if col not in ["Sequence_ID", "Sequence"]
]

missing = model_dataset[sequence_feature_cols].isna().sum()

print("Total sequence features:", len(sequence_feature_cols))
print("\nFeatures with missing values:")
print(missing[missing > 0])

print(
    "\nTotal missing sequence-feature values:",
    missing.sum()
)

Total sequence features: 43

Features with missing values:
Series([], dtype: int64)

Total missing sequence-feature values: 0


In [42]:
model_dataset.to_csv("LLPSDB_v2_model_dataset.csv", index=False)
print("Saved: LLPSDB_v2_model_dataset.csv")
print("Shape:", model_dataset.shape)

Saved: LLPSDB_v2_model_dataset.csv
Shape: (20719, 60)
